# Two-Tower Travel Recommender System

Pipeline hoàn chỉnh từ ETL đến đánh giá offline cho hệ thống gợi ý du lịch dựa trên Yelp dataset (2005–2022).

**Kiến trúc:** Two-Tower Retrieval với TensorFlow Recommenders (TFRS)  
**Môi trường:** Google Colab Pro  
**Dữ liệu:** ~1.2 triệu reviews, 32.981 businesses

## Version 15 improvements

This notebook continues from v14 and focuses on the next low-risk improvements from `docs/plan-enhance-two-tower.md`:

- query-aware attention pooling for user history, conditioned on selected city + trip intent + intent vibe;
- separate output/checkpoint directory `output_train_15` and experiment tag with `attn`;
- user-level evaluation now reports `NDCG@K` and `MRR@K`;
- slot coverage uses normalized/fuzzy category mapping so Vietnamese category variants do not collapse to zero;
- top-K evaluation uses `argpartition` before sorting to reduce evaluation time.


## Section 0 — Cài đặt môi trường

Cài đặt các thư viện cần thiết, khai báo đường dẫn dữ liệu, và định nghĩa class `Config` chứa toàn bộ hằng số cấu hình dùng xuyên suốt notebook. GPU memory growth được bật để tránh TensorFlow chiếm toàn bộ VRAM.

In [ ]:
# ── Cài đặt thư viện ──────────────────────────────────────────────────────────
# ScaNN bị loại bỏ: binary không tương thích với TF 2.16+ (absl ABI mismatch)
# → dùng tfrs BruteForce làm retrieval index thay thế
!pip uninstall -y scann 2>/dev/null || true
!pip install -q tensorflow-recommenders jsonlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 1.5 MB/s eta 0:00:00


In [ ]:
import os, sys
import pathlib

# Bat buoc dat TRUOC khi import tensorflow -- TFRS yeu cau Keras 2
os.environ['TF_USE_LEGACY_KERAS'] = '1'

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = pathlib.Path('/content/drive/MyDrive/train')
    SAVE_DIR = pathlib.Path('/content/drive/MyDrive/train/output_train_15')
except ImportError:
    IN_COLAB = False
    DATA_DIR = pathlib.Path('data/yelp')
    SAVE_DIR = pathlib.Path('output_train')

SAVE_DIR.mkdir(parents=True, exist_ok=True)
(SAVE_DIR / 'tb_logs').mkdir(parents=True, exist_ok=True)

# Thư mục lưu các tập dữ liệu đã split (ETL output)
ETL_CACHE_DIR = SAVE_DIR / 'etl_splits'
ETL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# --- Experiment tracking ---
# Chi can doi TEMPERATURE_SWEEP — EXPERIMENT_TAG va MODEL_SAVE_DIR tu dong cap nhat.
TEMPERATURE_SWEEP = 0.05   # <<< THAY GIA TRI NAY KHI SWEEP
_temp_str         = f'temp{int(TEMPERATURE_SWEEP * 10000):05d}'   # 0.05 -> 'temp00500'
EXPERIMENT_TAG    = f'dim256_hist30_attn_{_temp_str}'
MODEL_SAVE_DIR    = SAVE_DIR / EXPERIMENT_TAG
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
(MODEL_SAVE_DIR / 'tb_logs').mkdir(parents=True, exist_ok=True)

# File paths (Supabase UUID 'id', bge_embedding)
YELP_BUSINESS_FILE  = DATA_DIR / 'embedded_yelp_business_tourism.jsonl'
YELP_REVIEW_FILE    = DATA_DIR / 'two_tower_training_data_v3_renamed.jsonl'
FOODY_BUSINESS_FILE = DATA_DIR / 'embedded_foody_training_places.jsonl'
FOODY_REVIEW_FILE   = DATA_DIR / 'foody_two_tower_training_data_with_place_id.jsonl'

print(f'TF_USE_LEGACY_KERAS : {os.environ.get("TF_USE_LEGACY_KERAS")}')
print(f'IN_COLAB            : {IN_COLAB}')
print(f'DATA_DIR            : {DATA_DIR}')
print(f'SAVE_DIR            : {SAVE_DIR}')
print(f'EXPERIMENT_TAG      : {EXPERIMENT_TAG}')
print(f'MODEL_SAVE_DIR      : {MODEL_SAVE_DIR}')
print(f'ETL_CACHE_DIR       : {ETL_CACHE_DIR}')

Mounted at /content/drive
TF_USE_LEGACY_KERAS : 1
IN_COLAB            : True
DATA_DIR            : /content/drive/MyDrive/train
SAVE_DIR            : /content/drive/MyDrive/train/output_train_15
EXPERIMENT_TAG      : dim256_hist30_attn_temp00500
MODEL_SAVE_DIR      : /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500
ETL_CACHE_DIR       : /content/drive/MyDrive/train/output_train_15/etl_splits


In [ ]:
# Hang so cot loi
class Config:
    # Embedding dims
    USER_EMB_DIM        = 64
    CITY_EMB_DIM        = 16
    BIZ_EMB_DIM         = 64
    TYPE_EMB_DIM        = 16     # types / vibes
    TRAVEL_TYPE_EMB_DIM = 16     # travel_type (Candidate) & trip_intent (Query)
    OUTPUT_DIM          = 256    # dot-product space

    # Data
    BGE_DIM             = 1024
    MAX_HISTORY_LEN     = 30
    MAX_TRAIN_SAMPLES   = 100    # power-user cap
    USER_MIN_INTERACTIONS = 2     # v14+: include near-cold users with enough signal

    # Training
    BATCH_SIZE          = 2048    # v14+: more in-batch negatives if GPU RAM allows
    EPOCHS              = 50
    FINAL_EPOCHS        = EPOCHS  # train final model for the full budget after temperature selection
    LEARNING_RATE       = 1e-3
    WEIGHT_DECAY        = 1e-4
    DROPOUT_RATE        = 0.2
    TEMPERATURE         = TEMPERATURE_SWEEP  # In-batch Softmax scaling ? set in Cell 3

    # Weighting
    CATEGORY_WEIGHT_MODE = 'sqrt_clipped'
    CATEGORY_WEIGHT_CLIP = (0.2, 3.0)

    # Query history
    USE_RECENCY_WEIGHTED_HISTORY = True
    USE_QUERY_ATTENTION_HISTORY  = True
    HISTORY_ENCODER              = 'query_attention'

    # Eval
    TOP_K_LIST          = [10, 50, 100]

    # Inference quota (dia diem moi ngay)
    DAILY_QUOTA = {
        "attraction":    4,
        "restaurant":    3,
        "cafe":          2,
        "entertainment": 1,
    }

print('Config loaded:')
for k, v in vars(Config).items():
    if not k.startswith('_'):
        print(f'  {k} = {v}')


Config loaded:
  USER_EMB_DIM = 64
  CITY_EMB_DIM = 16
  BIZ_EMB_DIM = 64
  TYPE_EMB_DIM = 16
  TRAVEL_TYPE_EMB_DIM = 16
  OUTPUT_DIM = 256
  BGE_DIM = 1024
  MAX_HISTORY_LEN = 30
  MAX_TRAIN_SAMPLES = 100
  USER_MIN_INTERACTIONS = 2
  BATCH_SIZE = 2048
  EPOCHS = 50
  FINAL_EPOCHS = 50
  LEARNING_RATE = 0.001
  WEIGHT_DECAY = 0.0001
  DROPOUT_RATE = 0.2
  TEMPERATURE = 0.05
  CATEGORY_WEIGHT_MODE = sqrt_clipped
  CATEGORY_WEIGHT_CLIP = (0.2, 3.0)
  USE_RECENCY_WEIGHTED_HISTORY = True
  USE_QUERY_ATTENTION_HISTORY = True
  HISTORY_ENCODER = query_attention
  TOP_K_LIST = [10, 50, 100]
  DAILY_QUOTA = {'attraction': 4, 'restaurant': 3, 'cafe': 2, 'entertainment': 1}


In [ ]:
import tensorflow as tf

# ── Cấu hình GPU ──────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU detected: {len(gpus)} device(s) — memory growth enabled')
else:
    print('No GPU detected — running on CPU')

print(f'TensorFlow version: {tf.__version__}')

GPU detected: 1 device(s) — memory growth enabled
TensorFlow version: 2.20.0


## Section 1 — ETL & Chia tập dữ liệu

**Chiến lược phân chia:** Temporal 80-10-10 (sort theo thời gian)

**Lọc positive:** `sentiment_label == 'POS'` (v3 data) hoặc `stars >= 3` (fallback)

**Power-user cap 100:** Giữ 100 reviews gần nhất/user.

**`current_city`** = city của target item (không phải home city).

**`example_weight`** = inverse-frequency theo category — giải quyết majority class bias.

In [ ]:
# --- ETL Cache Check ---
# Kiểm tra etl_splits/ có đủ 4 file parquet chưa.
# Có đủ  → load thẳng, bỏ qua toàn bộ Cells 1a–1f.
# Thiếu  → chạy ETL đầy đủ từ Cell 1a.
import pandas as pd

_ETL_FILES = {
    'df_train':      ETL_CACHE_DIR / 'df_train.parquet',
    'df_val':        ETL_CACHE_DIR / 'df_val.parquet',
    'df_test':       ETL_CACHE_DIR / 'df_test.parquet',
    'df_candidates': ETL_CACHE_DIR / 'df_candidates.parquet',
}
_CACHE_LOADED = all(p.exists() for p in _ETL_FILES.values())

if _CACHE_LOADED:
    print('ETL Cache HIT — load từ etl_splits/, bỏ qua Cells 1a–1f.')
    df_train      = pd.read_parquet(_ETL_FILES['df_train'])
    df_val        = pd.read_parquet(_ETL_FILES['df_val'])
    df_test       = pd.read_parquet(_ETL_FILES['df_test'])
    df_candidates = pd.read_parquet(_ETL_FILES['df_candidates'])
    for _n, _d in [('df_train', df_train), ('df_val', df_val),
                   ('df_test', df_test), ('df_candidates', df_candidates)]:
        print(f'  {_n}: {_d.shape}')
else:
    _missing = [n for n, p in _ETL_FILES.items() if not p.exists()]
    print(f'ETL Cache MISS (thiếu: {_missing}) — chạy ETL đầy đủ từ Cell 1a.')

ETL Cache MISS (thiếu: ['df_train', 'df_val', 'df_test', 'df_candidates']) — chạy ETL đầy đủ từ Cell 1a.


In [ ]:
if globals().get("_CACHE_LOADED", False):
    print("Skip ETL cell: cache already loaded.")
else:
    # Cell 1a -- Load du lieu tu local (Yelp + Foody, normalize schema)
    import pandas as pd
    import numpy as np


    def load_and_normalize_business(filepath):
        """Load business JSONL, normalize: business_id->id, embedding->bge_embedding."""
        df = pd.read_json(filepath, lines=True)
        if 'business_id' in df.columns and 'id' not in df.columns:
            df = df.rename(columns={'business_id': 'id'})
        if 'embedding' in df.columns and 'bge_embedding' not in df.columns:
            df = df.rename(columns={'embedding': 'bge_embedding'})
        if 'travel_type' not in df.columns:
            df['travel_type'] = None
        return df


    def load_and_normalize_reviews(filepath):
        """Load review JSONL, normalize: business_id->id, parse date."""
        df = pd.read_json(filepath, lines=True)
        if 'business_id' in df.columns and 'id' not in df.columns:
            df = df.rename(columns={'business_id': 'id'})
        if 'date' in df.columns:
            if df['date'].dtype in ['int64', 'float64']:
                df['date'] = pd.to_datetime(df['date'], unit='ms')
            else:
                df['date'] = pd.to_datetime(df['date'])
        return df


    # Load Yelp
    print('Loading Yelp business ...')
    df_yelp_biz = load_and_normalize_business(YELP_BUSINESS_FILE)
    print(f'  yelp business: {df_yelp_biz.shape}')

    print('Loading Yelp reviews ...')
    df_yelp_rev = load_and_normalize_reviews(YELP_REVIEW_FILE)
    print(f'  yelp reviews : {df_yelp_rev.shape}')

    biz_list = [df_yelp_biz]
    rev_list = [df_yelp_rev]

    # Load Foody (neu file ton tai)
    if FOODY_BUSINESS_FILE.exists() and FOODY_REVIEW_FILE.exists():
        print('\nLoading Foody business ...')
        df_foody_biz = load_and_normalize_business(FOODY_BUSINESS_FILE)
        print(f'  foody business: {df_foody_biz.shape}')
        print('Loading Foody reviews ...')
        df_foody_rev = load_and_normalize_reviews(FOODY_REVIEW_FILE)
        print(f'  foody reviews : {df_foody_rev.shape}')
        biz_list.append(df_foody_biz)
        rev_list.append(df_foody_rev)
    else:
        print('\nFoody files not found -- using Yelp only.')

    # Merge
    df_biz = pd.concat(biz_list, ignore_index=True).drop_duplicates(subset=['id'])
    df_rev = pd.concat(rev_list, ignore_index=True)

    print(f'\nTotal businesses : {len(df_biz):,}  (unique id: {df_biz["id"].nunique():,})')
    print(f'Total reviews    : {len(df_rev):,}')
    print(df_biz[['id', 'name', 'city', 'category', 'travel_type']].head())
    print(df_rev[['user_id', 'id', 'stars', 'date', 'trip_intent', 'intent_vibe']].head())


Loading Yelp business ...
  yelp business: (32743, 24)
Loading Yelp reviews ...
  yelp reviews : (952822, 12)

Loading Foody business ...
  foody business: (29844, 32)
Loading Foody reviews ...
  foody reviews : (239184, 15)

Total businesses : 62,587  (unique id: 62,587)
Total reviews    : 1,192,006
                       id                      name           city  \
0  Pns2l4eNsfO8kk83dixA6A  Abby Rappoport, LAC, CMQ  Santa Barbara   
1  tUFrWirKiKi_TAnsVWINQQ                    Target         Tucson   
2  MTSW4McQd7CbVtyjqoe9mw        St Honore Pastries   Philadelphia   
3  mWMc6_wTdE0EUBKIGXDVfA  Perkiomen Valley Brewery     Green Lane   
4  CF33F8-E6oudUQ46HnavjQ            Sonic Drive-In   Ashland City   

              category        travel_type  
0  Thư giãn & Thể thao  Nghỉ dưỡng & Biển  
1    Mua sắm & Dịch vụ  Đô thị & Vui chơi  
2              Ẩm thực  Ẩm thực & Bản địa  
3              Ẩm thực  Ẩm thực & Bản địa  
4              Ẩm thực  Ẩm thực & Bản địa  
             

## Cell 1b — Thống nhất Schema (Yelp → Foody-canonical)

Hai nguồn dữ liệu có schema khác nhau — chuẩn hóa về cùng một schema trước khi train:

| Trường model cần | Foody có | Yelp có |
|---|---|---|
| `city` | `city_name` | `city` |
| `category` | `category_name` | `category` |
| `types` (list) | `type_name` (str đơn) | `types` (list) |
| `stars` | `average_rating` | `stars` |
| `vibes` (list) | `vibes` | — (không có) |

In [ ]:
if globals().get("_CACHE_LOADED", False):
    print("Skip ETL cell: cache already loaded.")
else:
    # Cell 1b -- Thong nhat schema: Yelp + Foody -> canonical (uu tien Foody)
    import numpy as np

    # Cac cot can thiet cho model sau khi thong nhat
    CANONICAL_COLS = [
        'id', 'name', 'city', 'category', 'types', 'vibes',
        'stars', 'review_count', 'travel_type',
        'bge_embedding', 'combined_text',
        'latitude', 'longitude', 'address', 'unified_id',
    ]


    def unify_business_schema(df):
        """
        Chuan hoa df_biz ve schema chung (uu tien ten cot Foody).
        Goi TRUOC khi merge voi reviews.
        """
        df = df.copy()

        # 1. city: city_name (Foody) > city (Yelp)
        if 'city_name' in df.columns:
            if 'city' in df.columns:
                df['city'] = df['city_name'].fillna(df['city'])
            else:
                df['city'] = df['city_name']
            df = df.drop(columns=['city_name'])

        # 2. category: category_name (Foody) > category (Yelp)
        if 'category_name' in df.columns:
            if 'category' in df.columns:
                df['category'] = df['category_name'].fillna(df['category'])
            else:
                df['category'] = df['category_name']
            df = df.drop(columns=['category_name'])

        # 3. types: type_name (Foody, str) -> wrap thanh list; types (Yelp, list) -> giu nguyen
        #    Sau khi concat: Foody rows co type_name str, Yelp rows co types list
        def merge_types(row):
            foody_type = row.get('type_name', None)
            yelp_types = row.get('types', None)
            # Foody: co type_name hop le -> wrap thanh list
            if isinstance(foody_type, str) and foody_type.strip():
                return [foody_type.strip()]
            # Yelp: co types la list hop le
            if isinstance(yelp_types, (list, np.ndarray)) and len(yelp_types) > 0:
                return [str(t) for t in yelp_types if str(t).strip()]
            return []

        df['types'] = df.apply(merge_types, axis=1)
        if 'type_name' in df.columns:
            df = df.drop(columns=['type_name'])

        # 4. stars: average_rating (Foody) > stars (Yelp)
        if 'average_rating' in df.columns:
            if 'stars' in df.columns:
                df['stars'] = df['average_rating'].fillna(df['stars'])
            else:
                df['stars'] = df['average_rating']
            df = df.drop(columns=['average_rating'])

        # 5. bge_embedding: doi ten tu 'embedding' neu can
        if 'embedding' in df.columns and 'bge_embedding' not in df.columns:
            df = df.rename(columns={'embedding': 'bge_embedding'})

        # 6. vibes: dam bao la list (co the la NaN sau concat)
        if 'vibes' in df.columns:
            df['vibes'] = df['vibes'].apply(
                lambda x: [str(v) for v in x if str(v).strip()]
                if isinstance(x, (list, np.ndarray)) else []
            )

        # 7. Fill NaN cho cac cot string
        for col in ['city', 'category', 'travel_type', 'combined_text', 'name', 'address']:
            if col in df.columns:
                df[col] = df[col].fillna('')

        # 8. Xoa cac cot operational/metadata khong can cho training
        drop_cols = [
            # Yelp-specific
            'attributes', 'hours', 'is_open', 'is_tourism',
            'categories', 'categories_vi', 'vibes_source', 'postal_code', 'state',
            # Foody-specific operational
            'is_approved', 'is_active', 'registered_date', 'vendor_id',
            'image_url', 'updated_at', 'city_id', 'source_id', 'source',
            'type_id', 'place_url', 'district_old', 'visit_duration',
            'open_time', 'close_time', 'description', 'open_hour_compressed',
        ]
        drop_cols = [c for c in drop_cols if c in df.columns]
        df = df.drop(columns=drop_cols)

        return df


    # Ap dung thong nhat schema
    df_biz = unify_business_schema(df_biz)

    # Kiem tra
    print('=== Schema sau khi thong nhat ===')
    for col in df_biz.columns:
        sample = df_biz[col].dropna().iloc[0] if df_biz[col].notna().any() else None
        if isinstance(sample, list):
            vtype = f'list  e.g. {repr(sample[:2])}'
        elif isinstance(sample, float):
            vtype = f'float = {sample:.2f}'
        else:
            vtype = f'{type(sample).__name__} = {repr(str(sample))[:50]}'
        null_pct = df_biz[col].isna().mean() * 100
        print(f'  {col:30s}: {vtype}  (null: {null_pct:.1f}%)')

    print(f'\nTotal businesses: {len(df_biz):,}')
    print(f'Columns: {list(df_biz.columns)}')

    # Guard: cac cot bat buoc khong duoc rong hoan toan
    for req in ['id', 'city', 'category', 'types', 'vibes', 'stars', 'bge_embedding', 'travel_type']:
        assert req in df_biz.columns, f"Missing required column: '{req}'"
    print('\nSchema validation PASSED -- tat ca cot can thiet deu co mat.')


=== Schema sau khi thong nhat ===
  name                          : str = 'Abby Rappoport, LAC, CMQ'  (null: 0.0%)
  address                       : str = '1616 Chapala St, Ste 2'  (null: 0.0%)
  city                          : str = 'Santa Barbara'  (null: 0.0%)
  latitude                      : float = 34.43  (null: 0.0%)
  longitude                     : float = -119.71  (null: 0.0%)
  stars                         : float = 5.00  (null: 0.0%)
  review_count                  : int64 = '7'  (null: 0.0%)
  category                      : str = 'Thư giãn & Thể thao'  (null: 0.0%)
  types                         : list  e.g. ['Spa & Thư giãn']  (null: 0.0%)
  vibes                         : list  e.g. ['Khách solo', 'Trong nhà']  (null: 0.0%)
  travel_type                   : str = 'Nghỉ dưỡng & Biển'  (null: 0.0%)
  combined_text                 : str = 'Abby Rappoport, LAC, CMQ. Spa & Thư giãn tại Sant  (null: 0.0%)
  unified_id                    : str = 'yelp_Pns2l4eNsfO8kk83dixA6A'

In [ ]:
if globals().get("_CACHE_LOADED", False):
    print("Skip ETL cell: cache already loaded.")
else:
    # Cell 1b -- Loc & Merge
    # Uu tien sentiment_label (v3 da loc san POS), fallback stars >= 3
    if 'sentiment_label' in df_rev.columns:
        df_rev_pos = df_rev[df_rev['sentiment_label'] == 'POS'].copy()
        print(f'Reviews after sentiment_label==POS: {len(df_rev_pos):,}')
    else:
        df_rev_pos = df_rev[df_rev['stars'] >= 3].copy()
        print(f'Reviews after stars >= 3: {len(df_rev_pos):,}')

    # Chi giu reviews co id ton tai trong business data
    valid_ids = set(df_biz['id'])
    df_rev_pos = df_rev_pos[df_rev_pos['id'].isin(valid_ids)]
    print(f'Reviews after id validation: {len(df_rev_pos):,}')

    # Doi ten stars tu review de tranh xung dot khi merge
    df_rev_pos = df_rev_pos.rename(columns={'stars': 'stars_review'})

    # Chon cot tu business -- giu travel_type
    biz_cols  = ['id', 'city', 'category', 'types', 'vibes', 'travel_type',
                 'stars', 'review_count', 'bge_embedding']
    available = [c for c in biz_cols if c in df_biz.columns]
    df_biz_sub = df_biz[available].rename(columns={'stars': 'stars_biz'})

    # Merge
    df = df_rev_pos.merge(df_biz_sub, on='id', how='inner')
    print(f'After merge shape: {df.shape}')
    print(df[['user_id', 'id', 'stars_review', 'city', 'category',
              'travel_type', 'trip_intent', 'intent_vibe']].head())


Reviews after sentiment_label==POS: 1,192,006
Reviews after id validation: 1,192,006
After merge shape: (1192006, 23)
                  user_id                      id  stars_review  \
0  ---1lKK3aKOuomHnwAkAow  f19eLfhXqR47Ct8Hz2y_pA           5.0   
1  ---2PmXbF47D870stH1jqA  eR7ieJD12PUzsYrP8fw6rQ           5.0   
2  ---2PmXbF47D870stH1jqA  igC3UWYb9RF5CXOQOVypMw           5.0   
3  ---2PmXbF47D870stH1jqA  hKameFsaXh9g8WQbv593UA           5.0   
4  ---2PmXbF47D870stH1jqA  ZvI9Ytqx_S_kahct3YzS0w           5.0   

            city             category        travel_type        trip_intent  \
0           Reno  Thư giãn & Thể thao  Nghỉ dưỡng & Biển  Khám phá tổng hợp   
1  Pinellas Park              Ẩm thực  Ẩm thực & Bản địa  Khám phá tổng hợp   
2     Clearwater              Ẩm thực  Ẩm thực & Bản địa  Ẩm thực & Bản địa   
3           Lutz              Ẩm thực  Ẩm thực & Bản địa  Ẩm thực & Bản địa   
4          Tampa              Ẩm thực  Ẩm thực & Bản địa  Ẩm thực & Bản địa   

    

In [ ]:
if globals().get("_CACHE_LOADED", False):
    print("Skip ETL cell: cache already loaded.")
else:
    # Cell 1c -- Power-user cap & Sap xep
    df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

    df = (
        df.groupby('user_id', group_keys=False)
        .apply(lambda g: g.tail(Config.MAX_TRAIN_SAMPLES))
        .reset_index(drop=True)
    )

    print(f'After power-user cap ({Config.MAX_TRAIN_SAMPLES}/user): {len(df):,} rows')
    print(f'Unique users     : {df["user_id"].nunique():,}')
    print(f'Unique businesses: {df["id"].nunique():,}')


/tmp/ipykernel_16165/3951506935.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.tail(Config.MAX_TRAIN_SAMPLES))


After power-user cap (100/user): 1,164,857 rows
Unique users     : 562,374
Unique businesses: 59,603


In [ ]:
if globals().get("_CACHE_LOADED", False):
    print("Skip ETL cell: cache already loaded.")
else:
    # Cell 1d -- Xay dung History Features (No-Leakage, dung 'id')
    import pathlib
    import pandas as pd

    HISTORY_CACHE = ETL_CACHE_DIR / 'df_with_history_v4.parquet'

    if HISTORY_CACHE.exists():
        print(f'Cache found -- loading from {HISTORY_CACHE}')
        df = pd.read_parquet(HISTORY_CACHE)
        print(f'Loaded. Shape: {df.shape}')
    else:
        print('No cache -- building sequential history features ...')

        def build_sequential_features(group):
            group = group.reset_index(drop=True)
            history_types_list = []
            history_vibes_list = []
            history_biz_list   = []
            accumulated_types  = []
            accumulated_vibes  = []
            accumulated_biz    = []

            for _, row in group.iterrows():
                # Luu lich su TAI THOI DIEM TRUOC row hien tai (strict left-exclusive)
                history_types_list.append(list(accumulated_types))
                history_vibes_list.append(list(accumulated_vibes))
                history_biz_list.append(list(accumulated_biz))

                # Cong don features cua row hien tai cho buoc tiep theo
                types_val = row.get('types', [])
                if not isinstance(types_val, (list, tuple)): types_val = []
                vibes_val = row.get('vibes', [])
                if not isinstance(vibes_val, (list, tuple)): vibes_val = []

                accumulated_types.extend(types_val)
                accumulated_vibes.extend(vibes_val)
                accumulated_biz.append(str(row['id']))  # dung UUID 'id'

            group['history_types']       = history_types_list
            group['history_vibes']       = history_vibes_list
            group['history_business_id'] = history_biz_list   # giu ten nay cho model
            group['current_city']        = group['city']
            return group

        df = (
            df.groupby('user_id', group_keys=False)
            .apply(build_sequential_features)
            .reset_index(drop=True)
        )

        # Ép kiểu các cột ID sang string để tránh lỗi mixed types của PyArrow
        if 'review_id' in df.columns:
            df['review_id'] = df['review_id'].astype(str)
        if 'user_id' in df.columns:
            df['user_id'] = df['user_id'].astype(str)

        df.to_parquet(HISTORY_CACHE, index=False)
        print(f'Cache saved to {HISTORY_CACHE}')

    print(f'Shape: {df.shape}')
    display_cols = ['id', 'trip_intent', 'intent_vibe', 'history_types', 'history_vibes']
    existing = [c for c in display_cols if c in df.columns]
    print(df[existing].head(5).to_string())


No cache -- building sequential history features ...


/tmp/ipykernel_16165/106342505.py:50: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_sequential_features)


Cache saved to /content/drive/MyDrive/train/output_train_15/etl_splits/df_with_history_v4.parquet
Shape: (1164857, 27)
                                     id        trip_intent intent_vibe                           history_types                                                                                                                                                                                                                                                                               history_vibes
0  8f495e5f-4630-5ffa-b3a8-7bd8f5c62034  Khám phá tổng hợp  Khách solo                                      []                                                                                                                                                                                                                                                                                          []
1  ed0acf41-19e3-53ac-a7c3-d592e53c2603  Ẩm thực & Bản địa  Khách solo                     

In [ ]:
if globals().get("_CACHE_LOADED", False):
    print("Skip ETL cell: cache already loaded.")
else:
    # Cell 1e — Temporal Split (80% Train - 10% Val - 10% Test)
    import pandas as pd

    train_rows, val_rows, test_rows = [], [], []

    print("Đang tiến hành chia tập dữ liệu theo mốc thời gian (Temporal Split)...")

    # df đã được sort theo 'user_id' và 'date'
    for uid, group in df.groupby('user_id'):
        group = group.reset_index(drop=True)
        n = len(group)

        # 1. Kịch bản Heavy Users (Đủ data để chia 80-10-10)
        if n >= 5:
            n_train = int(n * 0.8)
            # Đảm bảo Val và Test mỗi tập có ít nhất 1 mẫu
            n_val = max(1, int((n - n_train) / 2))

            train_rows.append(group.iloc[:n_train])
            val_rows.append(group.iloc[n_train : n_train + n_val])
            test_rows.append(group.iloc[n_train + n_val :])

        # 2. Kịch bản Moderate Users
        elif n >= 3:
            train_rows.append(group.iloc[:-2])
            val_rows.append(group.iloc[[-2]])
            test_rows.append(group.iloc[[-1]])

        # 3. Kịch bản Sparse Users
        elif n == 2:
            train_rows.append(group.iloc[[0]])
            val_rows.append(group.iloc[[1]])
        else:  # n == 1
            train_rows.append(group)

    df_train = pd.concat(train_rows, ignore_index=True)
    df_val   = pd.concat(val_rows,   ignore_index=True)
    df_test  = pd.concat(test_rows,  ignore_index=True)

    print(f'Train : {len(df_train):,} rows, {df_train["user_id"].nunique():,} users')
    print(f'Val   : {len(df_val):,} rows,   {df_val["user_id"].nunique():,} users')
    print(f'Test  : {len(df_test):,} rows,   {df_test["user_id"].nunique():,} users')

    # Kiểm tra tính toàn vẹn thời gian (Temporal Leakage Check)
    print("\nĐang kiểm tra rò rỉ dữ liệu tương lai...")
    train_last_date = df_train.groupby('user_id')['date'].max().rename('train_max_date')
    test_check = df_test[['user_id', 'date']].join(train_last_date, on='user_id')
    test_check = test_check.dropna(subset=['train_max_date'])

    # SỬA LỖI Ở ĐÂY: Dùng '<' thay vì '<=' do có user review nhiều địa điểm ở cùng 1 thời điểm tuyệt đối.
    violations = test_check[test_check['date'] < test_check['train_max_date']]
    assert len(violations) == 0, f'LỖI NGHIÊM TRỌNG: Phát hiện {len(violations)} dòng dữ liệu tương lai bị rò rỉ vào tập Train!'

    print('✅ KIỂM TRA THÀNH CÔNG: Toàn bộ dữ liệu Test đều xảy ra SAU HOẶC CÙNG LÚC với dữ liệu Train.')


Đang tiến hành chia tập dữ liệu theo mốc thời gian (Temporal Split)...
Train : 883,184 rows, 562,374 users
Val   : 183,180 rows,   168,952 users
Test  : 98,493 rows,   76,881 users

Đang kiểm tra rò rỉ dữ liệu tương lai...
✅ KIỂM TRA THÀNH CÔNG: Toàn bộ dữ liệu Test đều xảy ra SAU HOẶC CÙNG LÚC với dữ liệu Train.


In [ ]:
if globals().get("_CACHE_LOADED", False):
    print("Skip ETL cell: cache already loaded.")
else:
    # Kiểm tra số lượng review có cùng timestamp (cùng user_id và date) trong df_train
    dup_timestamps = df_train.groupby(['user_id', 'date']).size().reset_index(name='count')
    same_time_reviews = dup_timestamps[dup_timestamps['count'] > 1]

    print(f"Tổng số lần người dùng review nhiều địa điểm cùng 1 thời điểm: {len(same_time_reviews):,}")
    print(f"Số lượng user có thói quen review cùng lúc: {same_time_reviews['user_id'].nunique():,}")
    print(f"Tổng số review nằm trong các cụm cùng lúc này: {same_time_reviews['count'].sum():,}")
    print("\n--- Top 10 cụm review cùng lúc lớn nhất ---")
    display(same_time_reviews.sort_values('count', ascending=False).head(10))


Tổng số lần người dùng review nhiều địa điểm cùng 1 thời điểm: 44
Số lượng user có thói quen review cùng lúc: 43
Tổng số review nằm trong các cụm cùng lúc này: 91

--- Top 10 cụm review cùng lúc lớn nhất ---


,user_id,date,count
230859,7dBhV-vb0Yi_yf5--Hk6oA,2014-04-03 03:40:19.000,3
287774,9qMHsoxrTWvhZnozN5D6gg,2014-07-13 21:15:46.000,3
797165,sMdLDRyFeRHVK9qiNm6dnA,2012-02-15 14:12:06.000,3
7590,-fVoqxazbVktyNc2AUGX_g,2015-02-15 04:53:52.000,2
123171,2DOMvJe5EfvSHrKaOH-tTQ,2014-04-10 03:50:50.000,2
140434,3Ec1DHl3MEi3xfYoJivAfg,2012-06-04 19:58:44.000,2
172457,4LVjBOaErMhiwjMAbo1e2g,2014-11-10 01:08:05.000,2
146693,3ltfkfYn_cSPubFmc7BdTQ,2013-06-26 05:31:37.000,2
182613,535157,2019-03-20 15:40:04.053,2
198400,5hh180xTIOfRx_skElgEjQ,2013-06-03 19:30:04.000,2


In [ ]:
# Cell 1f -- Tinh example_weight & Luu tap du lieu
# v14: cell nay van chay khi ETL cache hit de upgrade/correct example_weight.
import numpy as np

if 'df_train' not in globals() or 'df_val' not in globals() or 'df_test' not in globals():
    raise RuntimeError('Thieu df_train/df_val/df_test. Hay chay ETL cache check va split cells truoc Cell 1f.')

# v14: sqrt-smoothed inverse-frequency weighting + clipping.
# Raw inverse-frequency was too extreme for a Foody/Yelp-like travel dataset
# and could under-train the majority food/local category.
category_counts = df_train['category'].value_counts()
total     = len(df_train)
n_classes = len(category_counts)

raw_weights = {
    cat: np.sqrt(total / (n_classes * cnt))
    for cat, cnt in category_counts.items()
}
mean_w = np.mean(list(raw_weights.values()))
category_weights = {cat: w / mean_w for cat, w in raw_weights.items()}
clip_min, clip_max = Config.CATEGORY_WEIGHT_CLIP

_weight_before_clip = df_train['category'].map(category_weights).fillna(1.0).astype('float32')
df_train['example_weight'] = _weight_before_clip.clip(clip_min, clip_max).astype('float32')
df_val['example_weight']  = 1.0
df_test['example_weight'] = 1.0

print('Category weights (sqrt-smoothed, clipped; cao = thieu so, thap = da so):')
print(df_train.groupby('category')['example_weight'].mean()
      .sort_values(ascending=False).to_string())
print(f'Weight range before clip: {_weight_before_clip.min():.4f} -> {_weight_before_clip.max():.4f}')
print(f'Weight range after  clip: {df_train["example_weight"].min():.4f} -> {df_train["example_weight"].max():.4f}')

# Luu parquet
df_train.to_parquet(ETL_CACHE_DIR / 'df_train.parquet', index=False)
df_val.to_parquet(ETL_CACHE_DIR / 'df_val.parquet',   index=False)
df_test.to_parquet(ETL_CACHE_DIR / 'df_test.parquet',  index=False)

# df_candidates = toan bo business data (da normalize).
# Cache hit: df_candidates da load o ETL Cache Check.
# Cache miss: tao tu df_biz nhu v13/v14 ban dau.
if 'df_candidates' not in globals():
    if 'df_biz' not in globals():
        raise RuntimeError('Thieu df_candidates/df_biz de luu candidate cache.')
    df_candidates = df_biz.copy()
    if 'stars' in df_candidates.columns and 'stars_biz' not in df_candidates.columns:
        df_candidates = df_candidates.rename(columns={'stars': 'stars_biz'})
else:
    if 'stars' in df_candidates.columns and 'stars_biz' not in df_candidates.columns:
        df_candidates = df_candidates.rename(columns={'stars': 'stars_biz'})

df_candidates.to_parquet(ETL_CACHE_DIR / 'df_candidates.parquet', index=False)

print(f'\nSaved:')
print(f'  df_train     : {len(df_train):,} rows | {df_train["user_id"].nunique():,} users')
print(f'  df_val       : {len(df_val):,} rows')
print(f'  df_test      : {len(df_test):,} rows')
print(f'  df_candidates: {len(df_candidates):,} businesses')
print(f'  columns      : {list(df_candidates.columns)}')

# Temporal integrity check
train_last = df_train.groupby('user_id')['date'].max().rename('train_max')
test_check = df_test[['user_id', 'date']].join(train_last, on='user_id').dropna(subset=['train_max'])
violations = test_check[test_check['date'] < test_check['train_max']]
assert len(violations) == 0, f'TEMPORAL LEAKAGE: {len(violations)} rows!'
print('Temporal integrity check PASSED.')


Category weights (sqrt-smoothed, clipped; cao = thieu so, thap = da so):
category
Văn hóa & Di sản        2.125140
Tham quan & Khám phá    1.289686
Lưu trú                 1.060374
Giải trí & Vui chơi     0.994939
Thư giãn & Thể thao     0.704923
Mua sắm & Dịch vụ       0.652018
Ẩm thực                 0.200000
Weight range before clip: 0.1729 -> 2.1251
Weight range after  clip: 0.2000 -> 2.1251

Saved:
  df_train     : 883,184 rows | 562,374 users
  df_val       : 183,180 rows
  df_test      : 98,493 rows
  df_candidates: 62,587 businesses
  columns      : ['name', 'address', 'city', 'latitude', 'longitude', 'stars_biz', 'review_count', 'category', 'types', 'vibes', 'travel_type', 'combined_text', 'unified_id', 'bge_embedding', 'id']
Temporal integrity check PASSED.


## Section 2 — Xây dựng tf.data Pipeline

**Anti-leakage:** Vocabulary và Normalization chỉ `.adapt()` trên `df_train`.

**BGE-m3 (Frozen input):** `bge_embedding` đưa thẳng vào Gate 1 — không fine-tune encoder.

**travel_type:** Thêm vào cả training pair và candidate — Dual Encoder Alignment.

**example_weight:** Tính inverse-frequency theo category trong ETL, lưu vào parquet.

In [ ]:
# Cell 2a -- Build Vocabularies (chi tu df_train)
import pickle
import gc as _gc
import pandas as pd

# -- RAM Optimization: Kiem tra xem dataframes da ton tai trong memory chua --
# Neu chay lien mach tu Section 1, ta co the tai su dung truc tiep de tiet kiem RAM.
if 'df_train' not in globals() or 'df_val' not in globals() or 'df_test' not in globals() or 'df_candidates' not in globals():
    print('Loading data from parquet files...')
    df_train      = pd.read_parquet(ETL_CACHE_DIR / 'df_train.parquet')
    df_val        = pd.read_parquet(ETL_CACHE_DIR / 'df_val.parquet')
    df_test       = pd.read_parquet(ETL_CACHE_DIR / 'df_test.parquet')
    df_candidates = pd.read_parquet(ETL_CACHE_DIR / 'df_candidates.parquet')
else:
    print('Using dataframes already in memory from previous cells.')

# -- RAM Optimization: xoa cot embedding khoi train/val/test --
# bge_embedding duoc luu trung lap trong moi row do buoc merge voi business.
# Thay vao do, pipeline se xay dung 1 dict lookup tu df_candidates (< 300MB).
_EMB_COLS = ['bge_embedding', 'embedding']
for _n, _d in [('df_train', df_train), ('df_val', df_val), ('df_test', df_test)]:
    _cols = [c for c in _EMB_COLS if c in _d.columns]
    if _cols:
        _mb_before = _d.memory_usage(deep=True).sum() / 1e6
        _d.drop(columns=_cols, inplace=True)
        _mb_after  = _d.memory_usage(deep=True).sum() / 1e6
        print(f'  RAM: {_n} dropped {_cols} | {_mb_before:.0f}MB -> {_mb_after:.0f}MB')
_gc.collect()


print('Building vocabularies from df_train ONLY ...')
print(f'User vocab threshold: >= {Config.USER_MIN_INTERACTIONS} train interactions')

def flatten_list_col(series):
    result = set()
    for lst in series:
        if lst is None:
            continue
        try:
            for v in lst:
                if v is not None and v == v and str(v).strip():
                    result.add(str(v))
        except TypeError:
            pass
    return sorted(result)


# Vocabulary chuan cua 5 travel types + Kham pha tong hop
KNOWN_TRAVEL_TYPES = [
    "Ẩm thực & Bản địa",
    "Đô thị & Vui chơi",
    "Khám phá & Sinh thái",
    "Khám phá tổng hợp",
    "Nghỉ dưỡng & Biển",
    "Văn hóa & Lịch sử",
]

user_counts = df_train['user_id'].value_counts()
loyal_users = user_counts[user_counts >= Config.USER_MIN_INTERACTIONS].index.tolist()

# travel_type vocab = known list + bat ky gia tri nao co trong data
data_tt = []
for src in [df_candidates, df_train]:
    if 'travel_type' in src.columns:
        data_tt += src['travel_type'].dropna().unique().tolist()
travel_type_vocab = sorted(set(KNOWN_TRAVEL_TYPES + [str(v) for v in data_tt if v]))

vocab = {
    'user_id'     : sorted(loyal_users),
    'business_id' : sorted(df_train['id'].unique().tolist()),
    'city'        : sorted(df_train['current_city'].dropna().unique().tolist()),
    'category'    : sorted(df_train['category'].dropna().unique().tolist()),
    'types'       : flatten_list_col(df_train['types']),
    'vibes'       : flatten_list_col(df_train['vibes']),
    'trip_intent' : sorted(df_train['trip_intent'].dropna().unique().tolist()),
    'intent_vibe' : sorted(df_train['intent_vibe'].dropna().unique().tolist()),
    'travel_type' : travel_type_vocab,
}

with open(SAVE_DIR / 'vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

for k, v in vocab.items():
    print(f'  vocab[{k:14s}]: {len(v):,} unique values')

for key in ('types', 'vibes', 'travel_type'):
    assert len(vocab[key]) > 0, f"vocab['{key}'] is empty!"
print('vocab.pkl saved.')

# -- SBC (Sampling-Bias Correction) --
# Tinh log-frequency cho moi business_id trong vocab.
# Dung de correct logit theo paper: Yi et al. 2019.
import numpy as np
from collections import Counter

_biz_counts = Counter(df_train['id'].astype(str).tolist())
_total      = len(df_train)

vocab['biz_log_freq'] = [
    float(np.log(max(_biz_counts.get(bid, 1), 1) / _total))
    for bid in vocab['business_id']
]

# Re-save vocab voi biz_log_freq
with open(SAVE_DIR / 'vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

_lf = vocab['biz_log_freq']
print(f'SBC freq computed: {len(_lf):,} items | log_freq [{min(_lf):.3f}, {max(_lf):.3f}]')


Using dataframes already in memory from previous cells.
  RAM: df_train dropped ['bge_embedding'] | 10940MB -> 3112MB
  RAM: df_val dropped ['bge_embedding'] | 2308MB -> 684MB
  RAM: df_test dropped ['bge_embedding'] | 1300MB -> 427MB
Building vocabularies from df_train ONLY ...
User vocab threshold: >= 2 train interactions
  vocab[user_id       ]: 54,580 unique values
  vocab[business_id   ]: 56,020 unique values
  vocab[city          ]: 838 unique values
  vocab[category      ]: 7 unique values
  vocab[types         ]: 36 unique values
  vocab[vibes         ]: 18 unique values
  vocab[trip_intent   ]: 6 unique values
  vocab[intent_vibe   ]: 4 unique values
  vocab[travel_type   ]: 6 unique values
vocab.pkl saved.
SBC freq computed: 56,020 items | log_freq [-13.691, -5.873]


In [ ]:
# Cell 2b & 2c -- Tao tf.data Pipeline (RAM-optimized)
import time
import gc
import numpy as np
import tensorflow as tf

AUTOTUNE    = tf.data.AUTOTUNE
BS          = Config.BATCH_SIZE
MAX_SEQ_LEN = Config.MAX_HISTORY_LEN


def pad_sequence(lst, max_len, pad_value=''):
    if not isinstance(lst, (list, np.ndarray)):
        lst = []
    lst = list(lst)[-max_len:]
    if len(lst) < max_len:
        lst = [pad_value] * (max_len - len(lst)) + lst
    return lst


# -- RAM Optimization 1: Embedding Lookup Dict --
# Xay dung 1 dict {business_id -> float32[1024]} tu df_candidates thay vi doc
# trung lap tu moi row cua df_train/val/test. Tiet kiem ~3-4GB RAM.
print('Building embedding lookup dict from df_candidates ...')
_t0 = time.time()
_emb_lookup = {}
for _, _row in df_candidates.iterrows():
    _bid = str(_row.get('id', _row.get('business_id', '')))
    _bge = _row.get('bge_embedding', _row.get('embedding', None))
    if _bge is not None and _bid:
        _emb_lookup[_bid] = np.array(_bge, dtype=np.float32)
_emb_bytes = sum(v.nbytes for v in _emb_lookup.values())
print(f'  Lookup: {len(_emb_lookup):,} items | {_emb_bytes/1e6:.0f}MB | {time.time()-_t0:.1f}s')


def create_tf_dataset(df, batch_size, is_training=True, is_candidate=False):
    if is_training:
        df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

    def generator():
        for _, row in df.iterrows():
            record = {}

            # User features (training/val/test, khong co trong candidate)
            if not is_candidate:
                record['user_id']             = str(row['user_id'])
                record['current_city']        = str(row.get('current_city', ''))
                record['trip_intent']         = str(row.get('trip_intent', ''))
                record['intent_vibe']         = str(row.get('intent_vibe', ''))
                record['history_types']       = pad_sequence(row.get('history_types', []),  MAX_SEQ_LEN)
                record['history_vibes']       = pad_sequence(row.get('history_vibes', []),  MAX_SEQ_LEN)
                record['history_business_id'] = pad_sequence(row.get('history_business_id', []), MAX_SEQ_LEN)
                record['example_weight']      = float(row.get('example_weight', 1.0))

            # Item features
            record['business_id']  = str(row.get('id', row.get('business_id', '')))
            record['city']         = str(row.get('city', ''))
            record['category']     = str(row.get('category', ''))
            record['travel_type']  = str(row.get('travel_type') or '')
            record['types']        = pad_sequence(row.get('types', []),  MAX_SEQ_LEN)
            record['vibes']        = pad_sequence(row.get('vibes', []),  MAX_SEQ_LEN)
            record['stars_biz']    = float(row.get('stars_biz', 0.0))
            record['review_count'] = float(row.get('review_count', 0.0))

            # -- RAM Optimization: doc embedding theo nguon --
            if is_candidate:
                # ds_candidates: doc truc tiep tu row (df_candidates giu embedding)
                _bge = row.get('bge_embedding', row.get('embedding', None))
                record['semantic_emb'] = (np.array(_bge, dtype=np.float32)
                                          if _bge is not None
                                          else np.zeros(Config.BGE_DIM, np.float32))
            else:
                # ds_train/val/test: look up tu _emb_lookup (tranh trung lap ~3-4GB)
                record['semantic_emb'] = _emb_lookup.get(
                    record['business_id'], np.zeros(Config.BGE_DIM, np.float32))
            yield record

    # Output signature
    sig = {}
    if not is_candidate:
        sig.update({
            'user_id':             tf.TensorSpec(shape=(), dtype=tf.string),
            'current_city':        tf.TensorSpec(shape=(), dtype=tf.string),
            'trip_intent':         tf.TensorSpec(shape=(), dtype=tf.string),
            'intent_vibe':         tf.TensorSpec(shape=(), dtype=tf.string),
            'history_types':       tf.TensorSpec(shape=(MAX_SEQ_LEN,), dtype=tf.string),
            'history_vibes':       tf.TensorSpec(shape=(MAX_SEQ_LEN,), dtype=tf.string),
            'history_business_id': tf.TensorSpec(shape=(MAX_SEQ_LEN,), dtype=tf.string),
            'example_weight':      tf.TensorSpec(shape=(), dtype=tf.float32),
        })
    sig.update({
        'business_id':  tf.TensorSpec(shape=(), dtype=tf.string),
        'city':         tf.TensorSpec(shape=(), dtype=tf.string),
        'category':     tf.TensorSpec(shape=(), dtype=tf.string),
        'travel_type':  tf.TensorSpec(shape=(), dtype=tf.string),
        'types':        tf.TensorSpec(shape=(MAX_SEQ_LEN,), dtype=tf.string),
        'vibes':        tf.TensorSpec(shape=(MAX_SEQ_LEN,), dtype=tf.string),
        'stars_biz':    tf.TensorSpec(shape=(), dtype=tf.float32),
        'review_count': tf.TensorSpec(shape=(), dtype=tf.float32),
        'semantic_emb': tf.TensorSpec(shape=(Config.BGE_DIM,), dtype=tf.float32),
    })

    ds = tf.data.Dataset.from_generator(generator, output_signature=sig)
    if is_training:
        # -- RAM Optimization 2: shuffle buffer 1000 (tu 2000, it RAM hon) --
        ds = ds.shuffle(buffer_size=1000, seed=42)
    ds = ds.batch(batch_size)

    # -- RAM Optimization 3: KHONG cache ds_train --
    # .cache() giu toan bo batches trong RAM (~3-4GB cho training set).
    # Chi cache val/test/candidates (nho, can iterate nhieu lan khi eval).
    if not is_training:
        ds = ds.cache()

    if is_training:
        ds = ds.repeat()
    return ds.prefetch(AUTOTUNE)


print('Building tf.data pipelines ...')
t0 = time.time()
ds_train      = create_tf_dataset(df_train,      BS, is_training=True,  is_candidate=False)
ds_val        = create_tf_dataset(df_val,        BS, is_training=False, is_candidate=False)
ds_test       = create_tf_dataset(df_test,       BS, is_training=False, is_candidate=False)
ds_candidates = create_tf_dataset(df_candidates, BS, is_training=False, is_candidate=True)
print(f'Pipelines built in {time.time()-t0:.1f}s')

sample = next(iter(ds_train))
print('\nSample batch shapes (ds_train):')
for k, v in sample.items():
    print(f'  {k:25s}: {v.shape}')


Building embedding lookup dict from df_candidates ...
  Lookup: 62,587 items | 256MB | 7.3s
Building tf.data pipelines ...
Pipelines built in 8.4s

Sample batch shapes (ds_train):
  user_id                  : (2048,)
  current_city             : (2048,)
  trip_intent              : (2048,)
  intent_vibe              : (2048,)
  history_types            : (2048, 30)
  history_vibes            : (2048, 30)
  history_business_id      : (2048, 30)
  example_weight           : (2048,)
  business_id              : (2048,)
  city                     : (2048,)
  category                 : (2048,)
  travel_type              : (2048,)
  types                    : (2048, 30)
  vibes                    : (2048, 30)
  stars_biz                : (2048,)
  review_count             : (2048,)
  semantic_emb             : (2048, 1024)


## Section 3 — Kiến trúc Model

### Query Tower
`user_id(64)` + `current_city(16)` + `trip_intent(16)` + `intent_vibe(8)`
+ `history_biz(64,pool)` + `history_types(16,pool)` + `history_vibes(16,pool)`
→ **Dense(256,relu) → Dropout(0.2) → Dense(128,relu) → Dropout(0.1) → Dense(128) → L2Norm**

### Candidate Tower (3 Gates Song Song)
- **Gate 1 — Semantic:** BGE-m3 1024d (frozen input) → Dense(256,relu) → Dense(128) → BN
- **Gate 2 — Categorical:** id + city + category + types + vibes + **travel_type** → Dense(128,relu) → BN
- **Gate 3 — Numerical:** stars_biz + log(1+review_count) → Normalization → Dense(32,relu) → BN
- **Fusion:** concat(128+128+32) → Dense(256,relu) → Dropout(0.2) → Dense(128) → L2Norm

### Lý do thiết kế
- **BGE-m3 frozen input + trainable projection:** tránh catastrophic forgetting
- **travel_type ⇔ trip_intent:** Dual Encoder Alignment — dot product học X↔Y alignment
- **log(review_count):** normalize long-tail distribution
- **compute_metrics=not training:** bỏ qua FactorizedTopK trong training batch → nhanh hơn ~3x
- **L2Normalize cả 2 towers:** dot product = cosine similarity ∈ [-1,1]

In [ ]:
# Cell 3 -- Kien truc Two-Tower
import tensorflow_recommenders as tfrs
import pickle
import tensorflow as tf

with open(SAVE_DIR / 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

EMB     = Config.OUTPUT_DIM   # 128
EMB_REG = tf.keras.regularizers.l2(1e-5)


# 0. Safe pooling chong NaN tu sequence rong + history attention.
class RecencyWeightedAveragePooling1D(tf.keras.layers.Layer):
    def __init__(self, recency_weighted=True, **kwargs):
        super().__init__(**kwargs)
        self.recency_weighted = recency_weighted
        self.supports_masking = True

    def call(self, inputs, mask=None):
        seq_len = tf.shape(inputs)[1]
        if self.recency_weighted:
            # pad_sequence left-pads history, so later positions are more recent.
            positions = tf.cast(tf.range(seq_len), inputs.dtype)
            weights = tf.exp(positions / tf.maximum(tf.cast(seq_len - 1, inputs.dtype), 1.0))
            weights = tf.reshape(weights, (1, seq_len, 1))
        else:
            weights = tf.ones((1, seq_len, 1), dtype=inputs.dtype)

        if mask is not None:
            weights = weights * tf.cast(tf.expand_dims(mask, -1), inputs.dtype)

        return tf.math.divide_no_nan(
            tf.reduce_sum(inputs * weights, axis=1),
            tf.reduce_sum(weights, axis=1)
        )


class QueryAwareAttentionPooling1D(tf.keras.layers.Layer):
    """Attention pooling for history, conditioned on current trip context."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True
        self.context_proj = None
        self.score_dense = tf.keras.layers.Dense(1)

    def build(self, input_shape):
        seq_dim = int(input_shape[0][-1])
        self.context_proj = tf.keras.layers.Dense(seq_dim, use_bias=False)
        super().build(input_shape)

    def call(self, inputs, mask=None):
        seq_emb, context_emb = inputs
        context_v = tf.expand_dims(self.context_proj(context_emb), axis=1)
        logits = tf.squeeze(self.score_dense(tf.nn.tanh(seq_emb + context_v)), axis=-1)

        seq_mask = None
        if isinstance(mask, (list, tuple)):
            seq_mask = mask[0]
        elif mask is not None:
            seq_mask = mask

        if seq_mask is not None:
            valid = tf.cast(seq_mask, logits.dtype)
            logits = tf.where(tf.cast(seq_mask, tf.bool), logits, tf.constant(-1e9, dtype=logits.dtype))
            weights = tf.nn.softmax(logits, axis=1) * valid
            weights = tf.math.divide_no_nan(weights, tf.reduce_sum(weights, axis=1, keepdims=True))
        else:
            weights = tf.nn.softmax(logits, axis=1)

        return tf.reduce_sum(seq_emb * tf.expand_dims(weights, -1), axis=1)


# Shared lookup/embedding layers (Query & Candidate dung chung weights)
shared_biz_lookup  = tf.keras.layers.StringLookup(vocabulary=vocab['business_id'],  mask_token='')
shared_biz_emb     = tf.keras.layers.Embedding(
    len(vocab['business_id']) + 2, Config.BIZ_EMB_DIM,
    mask_zero=True, embeddings_regularizer=EMB_REG)
shared_city_lookup = tf.keras.layers.StringLookup(vocabulary=vocab['city'], mask_token='')
shared_city_emb    = tf.keras.layers.Embedding(len(vocab['city']) + 2, Config.CITY_EMB_DIM)
shared_types_lookup= tf.keras.layers.StringLookup(vocabulary=vocab['types'], mask_token='')
shared_types_emb   = tf.keras.layers.Embedding(
    len(vocab['types']) + 2, Config.TYPE_EMB_DIM, mask_zero=True)
shared_vibes_lookup= tf.keras.layers.StringLookup(vocabulary=vocab['vibes'], mask_token='')
shared_vibes_emb   = tf.keras.layers.Embedding(
    len(vocab['vibes']) + 2, Config.TYPE_EMB_DIM, mask_zero=True)


# ==========================================================================
# 1. QUERY TOWER
# ==========================================================================
class QueryTower(tf.keras.Model):
    def __init__(self, vocab):
        super().__init__()
        self.user_lookup = tf.keras.layers.StringLookup(vocabulary=vocab['user_id'], mask_token='')
        self.user_emb    = tf.keras.layers.Embedding(
            len(vocab['user_id']) + 2, Config.USER_EMB_DIM, embeddings_regularizer=EMB_REG)

        self.city_lookup  = shared_city_lookup
        self.city_emb     = shared_city_emb
        self.biz_lookup   = shared_biz_lookup
        self.biz_emb      = shared_biz_emb
        self.types_lookup = shared_types_lookup
        self.types_emb    = shared_types_emb
        self.vibes_lookup = shared_vibes_lookup
        self.vibes_emb    = shared_vibes_emb
        self.recency_pool = RecencyWeightedAveragePooling1D(Config.USE_RECENCY_WEIGHTED_HISTORY)
        self.hbiz_pool    = QueryAwareAttentionPooling1D(name='history_business_attention')
        self.htype_pool   = QueryAwareAttentionPooling1D(name='history_type_attention')
        self.hvibe_pool   = QueryAwareAttentionPooling1D(name='history_vibe_attention')

        # trip_intent & intent_vibe -- mirror vocab voi candidate tower
        self.intent_lookup = tf.keras.layers.StringLookup(vocabulary=vocab['trip_intent'], mask_token='')
        self.intent_emb    = tf.keras.layers.Embedding(
            len(vocab['trip_intent']) + 2, Config.TRAVEL_TYPE_EMB_DIM)
        self.vibe_lookup   = tf.keras.layers.StringLookup(vocabulary=vocab['intent_vibe'], mask_token='')
        self.vibe_emb      = tf.keras.layers.Embedding(len(vocab['intent_vibe']) + 2, 8)

        # MLP sau hon theo guideline
        self.dense1  = tf.keras.layers.Dense(256, activation='relu')
        self.drop1   = tf.keras.layers.Dropout(Config.DROPOUT_RATE)   # 0.2
        self.dense2  = tf.keras.layers.Dense(128, activation='relu')
        self.drop2   = tf.keras.layers.Dropout(0.1)
        self.out     = tf.keras.layers.Dense(EMB)

    def call(self, inputs, training=False):
        user_v   = self.user_emb(self.user_lookup(inputs['user_id']))
        city_v   = self.city_emb(self.city_lookup(inputs['current_city']))
        intent_v = self.intent_emb(self.intent_lookup(inputs['trip_intent']))
        vibe_v   = self.vibe_emb(self.vibe_lookup(inputs['intent_vibe']))

        history_context = tf.concat([city_v, intent_v, vibe_v], axis=-1)

        hbiz_tok  = self.biz_lookup(inputs['history_business_id'])
        htype_tok = self.types_lookup(inputs['history_types'])
        hvibe_tok = self.vibes_lookup(inputs['history_vibes'])
        hbiz_seq  = self.biz_emb(hbiz_tok)
        htype_seq = self.types_emb(htype_tok)
        hvibe_seq = self.vibes_emb(hvibe_tok)

        if Config.USE_QUERY_ATTENTION_HISTORY:
            hbiz_v = self.hbiz_pool([hbiz_seq, history_context], mask=[tf.not_equal(hbiz_tok, 0), None])
            htype_v = self.htype_pool([htype_seq, history_context], mask=[tf.not_equal(htype_tok, 0), None])
            hvibe_v = self.hvibe_pool([hvibe_seq, history_context], mask=[tf.not_equal(hvibe_tok, 0), None])
        else:
            hbiz_v  = self.recency_pool(hbiz_seq, mask=tf.not_equal(hbiz_tok, 0))
            htype_v = self.recency_pool(htype_seq, mask=tf.not_equal(htype_tok, 0))
            hvibe_v = self.recency_pool(hvibe_seq, mask=tf.not_equal(hvibe_tok, 0))

        x = tf.concat([user_v, city_v, intent_v, vibe_v,
                       hbiz_v, htype_v, hvibe_v], axis=-1)
        x = self.drop1(self.dense1(x), training=training)
        x = self.drop2(self.dense2(x), training=training)
        return tf.math.l2_normalize(self.out(x), axis=-1)


# ==========================================================================
# 2. CANDIDATE TOWER (3 cong song song)
# ==========================================================================
class CandidateTower(tf.keras.Model):
    """
    Gate 1 -- Semantic : BGE-m3 1024d (frozen input) -> Dense(256,relu) -> Dense(128)
    Gate 2 -- Categorical: id, city, category, types, vibes, travel_type -> Dense(128)
    Gate 3 -- Numerical : log(1+review_count), stars -> Normalization -> Dense(32)
    Fusion: concat -> Dense(256,relu) -> Dropout(0.2) -> Dense(128) -> L2Norm
    """
    def __init__(self, vocab):
        super().__init__()

        # Gate 1 -- Semantic (projection trainable, BGE embeddings frozen input)
        self.sem_dense1 = tf.keras.layers.Dense(256, activation='relu')
        self.sem_dense2 = tf.keras.layers.Dense(128)
        self.bn_gate1   = tf.keras.layers.BatchNormalization()

        # Gate 2 -- Categorical
        self.biz_lookup   = shared_biz_lookup
        self.biz_emb      = shared_biz_emb
        self.city_lookup  = shared_city_lookup
        self.city_emb     = shared_city_emb
        self.cat_lookup   = tf.keras.layers.StringLookup(vocabulary=vocab['category'], mask_token='')
        self.cat_emb      = tf.keras.layers.Embedding(len(vocab['category']) + 2, 8)
        self.types_lookup = shared_types_lookup
        self.types_emb    = shared_types_emb
        self.types_pool   = RecencyWeightedAveragePooling1D(recency_weighted=False)
        self.vibes_lookup = shared_vibes_lookup
        self.vibes_emb    = shared_vibes_emb
        self.vibes_pool   = RecencyWeightedAveragePooling1D(recency_weighted=False)
        # travel_type -- Dual Encoder Alignment voi trip_intent trong QueryTower
        self.tt_lookup  = tf.keras.layers.StringLookup(vocabulary=vocab['travel_type'], mask_token='')
        self.tt_emb     = tf.keras.layers.Embedding(
            len(vocab['travel_type']) + 2, Config.TRAVEL_TYPE_EMB_DIM)
        self.cat_dense  = tf.keras.layers.Dense(128, activation='relu')
        self.bn_gate2   = tf.keras.layers.BatchNormalization()

        # Gate 3 -- Numerical
        self.normalizer = tf.keras.layers.Normalization(axis=-1)
        self.num_dense  = tf.keras.layers.Dense(32, activation='relu')
        self.bn_gate3   = tf.keras.layers.BatchNormalization()

        # Fusion
        self.fusion_dense = tf.keras.layers.Dense(256, activation='relu')
        self.dropout      = tf.keras.layers.Dropout(Config.DROPOUT_RATE)
        self.out          = tf.keras.layers.Dense(EMB)

    def adapt_numerical(self, df_candidates):
        """Fit Normalization tren log(1+review_count) va stars_biz tu DataFrame."""
        import numpy as np
        stars = df_candidates['stars_biz'].fillna(0).values.astype(np.float32)
        rc    = np.log1p(df_candidates['review_count'].fillna(0).values).astype(np.float32)
        self.normalizer.adapt(np.stack([stars, rc], axis=-1))
        print(f'Normalization adapted on {len(stars):,} candidates.')

    def call(self, inputs, training=False):
        # Gate 1 -- Semantic (BGE-m3 frozen input -> trainable projection)
        g1 = self.bn_gate1(
            self.sem_dense2(self.sem_dense1(inputs['semantic_emb'])),
            training=training)

        # Gate 2 -- Categorical (them travel_type)
        biz_v  = self.biz_emb(self.biz_lookup(inputs['business_id']))
        city_v = self.city_emb(self.city_lookup(inputs['city']))
        cat_v  = self.cat_emb(self.cat_lookup(inputs['category']))
        type_v = self.types_pool(self.types_emb(self.types_lookup(inputs['types'])))
        vibe_v = self.vibes_pool(self.vibes_emb(self.vibes_lookup(inputs['vibes'])))
        tt_v   = self.tt_emb(self.tt_lookup(inputs['travel_type']))
        g2 = self.bn_gate2(
            self.cat_dense(tf.concat([biz_v, city_v, cat_v, type_v, vibe_v, tt_v], axis=-1)),
            training=training)

        # Gate 3 -- Numerical (log1p transform tren review_count)
        log_rc    = tf.math.log1p(inputs['review_count'])
        num_input = tf.stack([inputs['stars_biz'], log_rc], axis=-1)
        g3 = self.bn_gate3(self.num_dense(self.normalizer(num_input)), training=training)

        # Fusion
        x = tf.concat([g1, g2, g3], axis=-1)
        x = self.dropout(self.fusion_dense(x), training=training)
        return tf.math.l2_normalize(self.out(x), axis=-1)


# ==========================================================================
# 3. TWO-TOWER RETRIEVAL MODEL (voi SBC - Sampling-Bias Correction)
# ==========================================================================
class TwoTowerRetrievalModel(tfrs.Model):
    # SBC: subtract log(p_j) khoi logit khi training
    # (Yi et al. 2019 'Sampling-Bias-Corrected Neural Modeling ...')
    def __init__(self, query_tower, candidate_tower, ds_candidates, biz_log_freq):
        super().__init__()
        self.query_tower     = query_tower
        self.candidate_tower = candidate_tower

        # SBC lookup: business_id (string) -> index -> log sampling prob
        self._sbc_lookup = tf.keras.layers.StringLookup(
            vocabulary=vocab['business_id'], mask_token='', output_mode='int')
        # Index 0 = OOV/mask: dung neutral value log(1/total)
        _neutral = float(np.log(1.0 / max(len(biz_log_freq), 1)))
        _lf_pad  = np.concatenate([[_neutral], [_neutral], biz_log_freq]).astype(np.float32)
        self._log_freq_table = tf.Variable(
            tf.constant(_lf_pad, dtype=tf.float32),
            trainable=False, name='sbc_log_freq')

        self.task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=ds_candidates.map(candidate_tower)
            ),
            temperature=Config.TEMPERATURE,
        )

    def compute_loss(self, features, training=False):
        query_emb     = self.query_tower(features, training=training)
        candidate_emb = self.candidate_tower(features, training=training)

        # SBC correction chi ap dung khi training
        sampling_prob = None
        if training:
            indices       = self._sbc_lookup(features['business_id'])  # (B,)
            log_freq      = tf.gather(self._log_freq_table, indices)    # (B,)
            sampling_prob = tf.exp(log_freq)                            # (B,) in (0,1)

        return self.task(
            query_emb,
            candidate_emb,
            sample_weight=features.get('example_weight'),
            compute_metrics=not training,
            candidate_sampling_probability=sampling_prob,
        )


print('Architecture classes defined: QueryTower, CandidateTower, TwoTowerRetrievalModel')
print(f'  Temperature = {Config.TEMPERATURE}  |  Output dim = {Config.OUTPUT_DIM}')
print(f'  History encoder = {Config.HISTORY_ENCODER}')

Architecture classes defined: QueryTower, CandidateTower, TwoTowerRetrievalModel
  Temperature = 0.05  |  Output dim = 256
  History encoder = query_attention


In [ ]:
import numpy as np
# Cell 3b -- Khoi tao model
query_tower     = QueryTower(vocab)
candidate_tower = CandidateTower(vocab)

# Adapt Normalization tu DataFrame (nhanh, chinh xac toan bo candidate pool)
candidate_tower.adapt_numerical(df_candidates)

biz_log_freq = np.array(vocab['biz_log_freq'], dtype=np.float32)
model = TwoTowerRetrievalModel(query_tower, candidate_tower, ds_candidates, biz_log_freq)

# AdamW: weight_decay regularize toan bo weights
model.compile(optimizer=tf.keras.optimizers.AdamW(
    learning_rate=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY
))

# Warm-up -- build computation graph
for batch in ds_train.take(1):
    _ = model.compute_loss(batch, training=False)

q_params = query_tower.count_params()
c_params = candidate_tower.count_params()
print(f'QueryTower     params: {q_params:,}')
print(f'CandidateTower params: {c_params:,}')
print(f'Total          params: {q_params + c_params:,}')
print(f'Optimizer: AdamW(lr={Config.LEARNING_RATE}, weight_decay={Config.WEIGHT_DECAY})')

Normalization adapted on 62,587 candidates.
QueryTower     params: 7,214,515
CandidateTower params: 4,053,837
Total          params: 11,268,352
Optimizer: AdamW(lr=0.001, weight_decay=0.0001)


## Section 4 — Training

**Optimizer:** AdamW (lr=1e-3, weight_decay=1e-4) — regularize toàn bộ weights.

**Callbacks:**
- `ModelCheckpoint`: monitor `val_factorized_top_k/top_100_categorical_accuracy` (max)
- `ReduceLROnPlateau`: monitor `val_total_loss`, patience=3, factor=0.5, min_lr=1e-6
- `EarlyStopping`: monitor `val_total_loss`, **patience=7** — sparse data cần thêm thời gian
- `TensorBoard`: visualize training curves real-time

**Lý do patience=7:** User trung bình 1-2 reviews → gradient noisy → cần nhiều epoch hơn để ổn định.

In [ ]:
# Cell 4a -- Khai bao Callbacks
import datetime
import tensorflow as tf

CHECKPOINT_PATH = str(MODEL_SAVE_DIR / 'best_model.weights.h5')
TB_LOG_DIR      = str(MODEL_SAVE_DIR / 'tb_logs' / datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))

MONITOR_TOP100 = 'val_factorized_top_k/top_100_categorical_accuracy'
MONITOR_LOSS   = 'val_total_loss'

callbacks = [
    # 1. Luu best weights theo Top-100 accuracy
    tf.keras.callbacks.ModelCheckpoint(
        filepath          = CHECKPOINT_PATH,
        monitor           = MONITOR_TOP100,
        save_best_only    = True,
        save_weights_only = True,
        mode              = 'max',
        verbose           = 1,
    ),
    # 2. Giam LR khi val_loss chung lai
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor  = MONITOR_LOSS,
        factor   = 0.5,
        patience = 3,
        min_lr   = 1e-6,
        mode     = 'min',
        verbose  = 1,
    ),
    # 3. EarlyStopping patience=7 — monitor cung metric voi ModelCheckpoint
    tf.keras.callbacks.EarlyStopping(
        monitor              = MONITOR_TOP100,
        patience             = 7,
        restore_best_weights = True,
        mode                 = 'max',
        verbose              = 1,
    ),
    # 4. TensorBoard
    tf.keras.callbacks.TensorBoard(
        log_dir        = TB_LOG_DIR,
        histogram_freq = 0,
    ),
]

print(f'Checkpoint path : {CHECKPOINT_PATH}')
print(f'TensorBoard logs: {TB_LOG_DIR}')
print(f'Monitor Top-100 : {MONITOR_TOP100}')
print(f'Monitor Loss    : {MONITOR_LOSS}')
print(f'Callbacks ready : {len(callbacks)}')

Checkpoint path : /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500/best_model.weights.h5
TensorBoard logs: /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500/tb_logs/20260610-041755
Monitor Top-100 : val_factorized_top_k/top_100_categorical_accuracy
Monitor Loss    : val_total_loss
Callbacks ready : 4


In [ ]:
# Cell 4b ? Ch?y Training
# v14: train final model with full Config.EPOCHS after choosing the best temperature.
TRAIN_EPOCHS = Config.FINAL_EPOCHS

steps_per_epoch  = len(df_train) // Config.BATCH_SIZE
validation_steps = len(df_val)   // Config.BATCH_SIZE

history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=TRAIN_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks
)

# L?u final weights v?o MODEL_SAVE_DIR
FINAL_WEIGHTS_PATH = str(MODEL_SAVE_DIR / 'final_model.weights.h5')
model.save_weights(FINAL_WEIGHTS_PATH)
print(f'Final weights saved to: {FINAL_WEIGHTS_PATH}')

best_top100 = max(history.history.get(MONITOR_TOP100, [0]))
best_loss   = min(history.history.get(MONITOR_LOSS,   [999]))
epochs_run  = len(history.history.get(MONITOR_LOSS,   []))
print(f'Best val Top-100 accuracy : {best_top100:.4f}')
print(f'Best val loss             : {best_loss:.4f}')
print(f'Epochs run                : {epochs_run}')

# --- Ghi log experiment ?? so s?nh c?c l?n ch?y ---
import json as _json
result_log = {
    'experiment':              EXPERIMENT_TAG,
    'output_dim':              Config.OUTPUT_DIM,
    'history_len':             Config.MAX_HISTORY_LEN,
    'temperature':             Config.TEMPERATURE,
    'train_epochs':            TRAIN_EPOCHS,
    'batch_size':              Config.BATCH_SIZE,
    'user_min_interactions':   Config.USER_MIN_INTERACTIONS,
    'category_weight_mode':    Config.CATEGORY_WEIGHT_MODE,
    'category_weight_clip':    Config.CATEGORY_WEIGHT_CLIP,
    'recency_weighted_history': Config.USE_RECENCY_WEIGHTED_HISTORY,
    'query_attention_history': Config.USE_QUERY_ATTENTION_HISTORY,
    'history_encoder':         Config.HISTORY_ENCODER,
    'best_top100':             float(best_top100),
    'best_loss':               float(best_loss),
    'epochs_run':              epochs_run,
}
log_path = SAVE_DIR / 'experiment_log.jsonl'
with open(log_path, 'a', encoding='utf-8') as _f:
    _f.write(_json.dumps(result_log, ensure_ascii=False) + '\n')
print(f'\nLogged to: {log_path}')
print(f'Result   : {result_log}')


Epoch 1/50
431/431 [==============================] - ETA: 0s - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_100_categorical_accuracy: 0.0000e+00 - loss: 2426.3673 - regularization_loss: 0.0350 - total_loss: 2426.4023
Epoch 1: val_factorized_top_k/top_100_categorical_accuracy improved from -inf to 0.61555, saving model to /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500/best_model.weights.h5


431/431 [==============================] - 1007s 2s/step - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_100_categorical_accuracy: 0.0000e+00 - loss: 2425.8225 - regularization_loss: 0.0350 - total_loss: 2425.8575 - val_factorized_top_k/top_1_categorical_accuracy: 0.0319 - val_factorized_top_k/top_5_categorical_accuracy: 0.1556 - val_factorized_top_k/top_10_categorical_accuracy: 0.2359 - val_factorized_top_k/top_50_categorical_accuracy: 0.4828 - val_factorized_top_k/top_100_categorical_accuracy: 0.6156 - val_loss: 9762.7305 - val_regularization_loss: 0.0420 - val_total_loss: 9762.7725 - lr: 0.0010
Epoch 2/50
431/431 [==============================] - ETA: 0s - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - factor

Final weights saved to: /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500/final_model.weights.h5
Best val Top-100 accuracy : 0.6156
Best val loss             : 8796.4873
Epochs run                : 8

Logged to: /content/drive/MyDrive/train/output_train_15/experiment_log.jsonl
Result   : {'experiment': 'dim256_hist30_attn_temp00500', 'output_dim': 256, 'history_len': 30, 'temperature': 0.05, 'train_epochs': 50, 'batch_size': 2048, 'user_min_interactions': 2, 'category_weight_mode': 'sqrt_clipped', 'category_weight_clip': (0.2, 3.0), 'recency_weighted_history': True, 'query_attention_history': True, 'history_encoder': 'query_attention', 'best_top100': 0.6155526041984558, 'best_loss': 8796.4873046875, 'epochs_run': 8}


### So sánh kết quả Temperature Grid Search

Chạy cell dưới sau khi hoàn thành tất cả 5 lần sweep để chọn temperature tốt nhất.

In [ ]:
# So sánh tất cả experiments đã chạy
import json as _json
import pandas as pd

log_path = SAVE_DIR / 'experiment_log.jsonl'
if log_path.exists():
    rows = [_json.loads(l) for l in open(log_path, encoding='utf-8')]
    df_log = pd.DataFrame(rows).sort_values('best_top100', ascending=False)
    print(df_log[[
        'experiment', 'output_dim', 'history_len', 'temperature',
        'best_top100', 'best_loss', 'epochs_run'
    ]].to_string(index=False))
    best_exp = df_log.iloc[0]
    print(f'\n>>> Best: temperature={best_exp["temperature"]}, top100={best_exp["best_top100"]:.4f}')
    print(f'    EXPERIMENT_TAG to use for final run: {best_exp["experiment"]}')
else:
    print(f'Log not found: {log_path}')

                  experiment  output_dim  history_len  temperature  best_top100   best_loss  epochs_run
dim256_hist30_attn_temp00500         256           30         0.05     0.615553 8796.487305           8

>>> Best: temperature=0.05, top100=0.6156
    EXPERIMENT_TAG to use for final run: dim256_hist30_attn_temp00500


In [ ]:
# Cell 4c — Plot Learning Curves
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

hist = history.history
epochs_range = range(1, len(hist['total_loss']) + 1)

top100_val_key = 'val_factorized_top_k/top_100_categorical_accuracy'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Loss
# NOTE: Train Loss dung in-batch negatives (1024 items); Val Loss dung full corpus (41k items).
# Gap giua hai duong la dac trung tat yeu cua retrieval training — KHONG phai overfitting.
axes[0].plot(epochs_range, hist['total_loss'],     label='Train Loss (in-batch, N=1024)')
axes[0].plot(epochs_range, hist['val_total_loss'], label='Val Loss (full corpus, N=41k)')
axes[0].set_title(f'Training & Validation Loss  [temp={Config.TEMPERATURE}]\n(khac nhau do evaluation space khac nhau)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Subplot 2: Top-100 Accuracy (chi val — train acc luon=0 theo thiet ke)
# Train Top-100 Acc luon = 0 do compute_metrics=False trong training step (tiet kiem ~3x thoi gian)
if top100_val_key in hist:
    axes[1].plot(epochs_range, hist[top100_val_key], label='Val Top-100 Acc', color='orange')
axes[1].set_title(f'Val Top-100 Acc  [temp={Config.TEMPERATURE}]\n(Train metric disabled: compute_metrics=not training)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
curve_path = str(MODEL_SAVE_DIR / 'training_curves.png')
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Learning curves saved to: {curve_path}')

Learning curves saved to: /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500/training_curves.png


## Section 5 ? ??nh gi? Offline

**Quy tr?nh:** Load best weights ? encode candidates ? city-aware/global retrieval ? dot-product ranking.

**Metrics:**
- `Precision@K`, `Recall@K`, `HitRate@K` ? ID-based retrieval coverage.
- `MAP@K`, `NDCG@K`, `MRR@K` ? ranking quality, v? tr? item ??ng trong top-K.
- `Type_Precision@K`, `Vibe_Precision@K` ? semantic match.
- `TravelType_Align@K` ? % top-K thu?c ??ng travel type user ch?n.
- `SlotCoverage@K` ? top-K c? ?? attraction + restaurant sau khi normalize category.
- `RetrievalScope` v? `UserSegment` t?ch `city_aware/global`, `all/loyal/near_cold`; loyal d?ng ng??ng `Config.USER_MIN_INTERACTIONS`.


In [ ]:
# ─── Bootstrap: Fresh-Session Setup (Section 5 & 6) ─────────────────────────
# Chay cell nay khi bat dau session moi de eval hoac demo.
# Prerequisite (chi can chay 4 cell nay truoc):
#   Cell 03 (paths) → Cell 04 (Config) → Cell 05 (GPU) → Cell 22 (model def)
# Sau do: cell nay → 31 → 32 → 33 → 34  (eval offline)
#          hoac     cell nay → 36          (demo / inference)

import pickle, time
import numpy as np
import pandas as pd
import tensorflow as tf

_t0 = time.time()

def _pad_seq(lst, max_len, pad_value=""):
    if not isinstance(lst, (list, np.ndarray)): lst = []
    lst = list(lst)[-max_len:]
    if len(lst) < max_len: lst = [pad_value] * (max_len - len(lst)) + lst
    return lst

# ── 1. Vocab ─────────────────────────────────────────────────────────────────
if "vocab" not in globals() or not isinstance(globals().get("vocab"), dict):
    with open(SAVE_DIR / "vocab.pkl", "rb") as _f:
        vocab = pickle.load(_f)
    print(f"[1/5] vocab loaded ({len(vocab)} keys)")
else:
    print("[1/5] vocab already in memory")

# ── 2. DataFrames ─────────────────────────────────────────────────────────────
if "df_candidates" not in globals() or not isinstance(globals().get("df_candidates"), pd.DataFrame):
    df_candidates = pd.read_parquet(ETL_CACHE_DIR / "df_candidates.parquet")
    print(f"[2/5] df_candidates loaded: {df_candidates.shape}")
else:
    print("[2/5] df_candidates already in memory")

if "df_test" not in globals() or not isinstance(globals().get("df_test"), pd.DataFrame):
    df_test = pd.read_parquet(ETL_CACHE_DIR / "df_test.parquet")
    print(f"     df_test loaded: {df_test.shape}")
else:
    print("     df_test already in memory")

# ── 3. ds_candidates (tf.data) ───────────────────────────────────────────────
if "ds_candidates" not in globals():
    _id_col  = "id" if "id" in df_candidates.columns else "business_id"
    _emb_col = "bge_embedding" if "bge_embedding" in df_candidates.columns else "embedding"
    _ML = Config.MAX_HISTORY_LEN
    _cd = {
        "business_id":  tf.constant(df_candidates[_id_col].astype(str).values, dtype=tf.string),
        "city":         tf.constant(df_candidates["city"].astype(str).values, dtype=tf.string),
        "category":     tf.constant(df_candidates["category"].astype(str).values, dtype=tf.string),
        "travel_type":  tf.constant(df_candidates["travel_type"].fillna("").astype(str).values, dtype=tf.string),
        "types":        tf.constant([_pad_seq(t, _ML) for t in df_candidates["types"]], dtype=tf.string),
        "vibes":        tf.constant([_pad_seq(v, _ML) for v in df_candidates["vibes"]], dtype=tf.string),
        "stars_biz":    tf.constant(df_candidates["stars_biz"].fillna(0).astype(np.float32).values, dtype=tf.float32),
        "review_count": tf.constant(df_candidates["review_count"].fillna(0).astype(np.float32).values, dtype=tf.float32),
        "semantic_emb": tf.constant(np.vstack(df_candidates[_emb_col].values).astype(np.float32), dtype=tf.float32),
    }
    ds_candidates = (
        tf.data.Dataset.from_tensor_slices(_cd)
        .batch(Config.BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )
    print(f"[3/5] ds_candidates built: {len(df_candidates):,} candidates")
else:
    print("[3/5] ds_candidates already in memory")

# ── 4. Model towers ───────────────────────────────────────────────────────────
if not all(k in globals() for k in ["query_tower", "candidate_tower", "model"]):
    query_tower     = QueryTower(vocab)
    candidate_tower = CandidateTower(vocab)
    candidate_tower.adapt_numerical(df_candidates)
    biz_log_freq    = np.array(vocab["biz_log_freq"], dtype=np.float32)
    model = TwoTowerRetrievalModel(
        query_tower, candidate_tower, ds_candidates, biz_log_freq)
    model.compile(optimizer=tf.keras.optimizers.AdamW(
        learning_rate=Config.LEARNING_RATE,
        weight_decay=Config.WEIGHT_DECAY))

    # Warm-up: build computation graph manually by creating a dummy batch with ALL required keys
    _ML = Config.MAX_HISTORY_LEN
    dummy_features = {
        'user_id': tf.constant([''], dtype=tf.string),
        'current_city': tf.constant([''], dtype=tf.string),
        'trip_intent': tf.constant([''], dtype=tf.string),
        'intent_vibe': tf.constant([''], dtype=tf.string),
        'history_types': tf.constant([[''] * _ML], dtype=tf.string),
        'history_vibes': tf.constant([[''] * _ML], dtype=tf.string),
        'history_business_id': tf.constant([[''] * _ML], dtype=tf.string),

        'business_id': tf.constant([''], dtype=tf.string),
        'city': tf.constant([''], dtype=tf.string),
        'category': tf.constant([''], dtype=tf.string),
        'travel_type': tf.constant([''], dtype=tf.string),
        'types': tf.constant([[''] * _ML], dtype=tf.string),
        'vibes': tf.constant([[''] * _ML], dtype=tf.string),
        'stars_biz': tf.constant([0.0], dtype=tf.float32),
        'review_count': tf.constant([0.0], dtype=tf.float32),
        'semantic_emb': tf.constant(np.zeros((1, Config.BGE_DIM)), dtype=tf.float32),
    }
    _ = model.compute_loss(dummy_features, training=False)
    print("[4/5] Model built & graph initialized")
else:
    print("[4/5] Model already in memory")

# ── 5. Load best weights ──────────────────────────────────────────────────────
_wt_path = MODEL_SAVE_DIR / "best_model.weights.h5"
if not _wt_path.exists():
    # Fallback: tim file weights moi nhat trong SAVE_DIR/*
    _wt_all = sorted(SAVE_DIR.glob("*/best_model.weights.h5"),
                     key=lambda p: p.stat().st_mtime, reverse=True)
    if _wt_all:
        _wt_path = _wt_all[0]
        print(f"[5/5] [fallback] weights: {_wt_path}")
    else:
        raise FileNotFoundError(
            f"Khong tim thay best_model.weights.h5 trong {SAVE_DIR}\n"
            f"Hay chay training truoc (Cell 25-26) roi moi chay eval.")
else:
    print(f"[5/5] weights: {_wt_path}")

model.load_weights(str(_wt_path))
print(f"\nBootstrap xong trong {time.time()-_t0:.1f}s.")
print("San sang: Cell 31→32→33→34 (eval) hoac Cell 36 (demo)")


[1/5] vocab already in memory
[2/5] df_candidates already in memory
     df_test already in memory
[3/5] ds_candidates already in memory
[4/5] Model already in memory
[5/5] weights: /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500/best_model.weights.h5



Bootstrap xong trong 5.8s.
San sang: Cell 31→32→33→34 (eval) hoac Cell 36 (demo)


In [ ]:
# Cell 5a -- Encode Candidate Corpus (re-runnable doc lap)
# Prerequisite: Bootstrap cell (truoc cell nay) phai duoc chay truoc.
import time
import numpy as np

if "candidate_tower" not in globals() or "ds_candidates" not in globals():
    raise RuntimeError(
        "Thieu bien. Hay chay Bootstrap cell truoc cell nay.")

# Reload best weights (an toan khi re-run de dung dung checkpoint)
_wt_path = MODEL_SAVE_DIR / "best_model.weights.h5"
model.load_weights(str(_wt_path))
print(f"Weights loaded: {_wt_path}")

# Encode toan bo candidates
t0 = time.time()
candidate_embeddings = []
candidate_ids_list   = []

for batch in ds_candidates:
    embs = candidate_tower(batch, training=False).numpy()
    ids  = batch["business_id"].numpy()
    candidate_embeddings.append(embs)
    candidate_ids_list.append(ids)

candidate_embeddings = np.concatenate(candidate_embeddings, axis=0)  # (N, OUTPUT_DIM)
candidate_ids_np     = np.concatenate(candidate_ids_list, axis=0)    # (N,)

print(f"Candidate embeddings shape : {candidate_embeddings.shape}")
print(f"Candidate IDs shape        : {candidate_ids_np.shape}")
print(f"Encode time                : {time.time()-t0:.1f}s")


Weights loaded: /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500/best_model.weights.h5
Candidate embeddings shape : (62587, 256)
Candidate IDs shape        : (62587,)
Encode time                : 1.8s


In [ ]:
# Cell 5b — Build Retrieval Index (BruteForce via TFRS)
# ScaNN không dùng do ABI mismatch với TF 2.16+ — BruteForce đủ dùng với 41k candidates

MAX_K = max(Config.TOP_K_LIST)  # 100

# Build BruteForce index từ candidate embeddings đã encode ở cell 5a
brute_force = tfrs.layers.factorized_top_k.BruteForce(
    query_model=None,  # query đã được encode thủ công ở cell 5c
    k=MAX_K
)

# Populate index: truyền (business_id, embedding) để index biết cần trả về id nào
cand_ids_tensor = tf.constant(candidate_ids_np, dtype=tf.string)
cand_emb_tensor = tf.constant(candidate_embeddings, dtype=tf.float32)

# BruteForce.index nhận (identifiers, embeddings)
brute_force.index(cand_emb_tensor, identifiers=cand_ids_tensor)

def retrieve_top_k(query_vec, k):
    """Trả về (B, k) business_id strings."""
    scores, ids = brute_force(tf.constant(query_vec, dtype=tf.float32), k=k)
    return ids.numpy()  # (B, k) bytes

print(f'BruteForce index populated — {len(candidate_embeddings):,} candidates, Top-{MAX_K}')
print('Retrieval backend: tfrs.layers.factorized_top_k.BruteForce')

BruteForce index populated — 62,587 candidates, Top-100
Retrieval backend: tfrs.layers.factorized_top_k.BruteForce


In [ ]:
# Cell 5c -- User-level Evaluation (City-Aware + Global + Cold/Loyal Split + NDCG/MRR)
import time
import unicodedata
import numpy as np
import pandas as pd
import tensorflow as tf


def pad_sequence(lst, max_len, pad_value=''):
    if not isinstance(lst, (list, np.ndarray)): lst = []
    lst = list(lst)[-max_len:]
    if len(lst) < max_len: lst = [pad_value] * (max_len - len(lst)) + lst
    return lst


def normalize_text(value):
    text = str(value or '').lower().strip()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    return ' '.join(text.replace('&', ' ').replace('-', ' ').replace('_', ' ').split())


def category_to_slot(category):
    c = normalize_text(category)
    if not c:
        return ''
    if any(tok in c for tok in ['am thuc', 'an uong', 'restaurant', 'food', 'quan an', 'nha hang', 'cafe', 'coffee']):
        return 'restaurant'
    if any(tok in c for tok in ['van hoa', 'di san', 'lich su', 'tham quan', 'kham pha', 'attraction', 'tourism', 'museum', 'chua', 'den']):
        return 'attraction'
    if any(tok in c for tok in ['giai tri', 'vui choi', 'bar', 'pub', 'club', 'cinema']):
        return 'entertainment'
    if any(tok in c for tok in ['mua sam', 'dich vu', 'shopping', 'mall', 'market']):
        return 'shopping'
    if any(tok in c for tok in ['thu gian', 'the thao', 'spa', 'relax', 'sport']):
        return 'relaxation'
    if any(tok in c for tok in ['luu tru', 'hotel', 'resort', 'homestay']):
        return 'accommodation'
    return ''


REQUIRED_SLOTS = {'attraction', 'restaurant'}


def _empty_metric_acc(top_k_list):
    return {
        'hit_rate':  {k: 0.0 for k in top_k_list},
        'recall':    {k: 0.0 for k in top_k_list},
        'precision': {k: 0.0 for k in top_k_list},
        'map_k':     {k: 0.0 for k in top_k_list},
        'ndcg':      {k: 0.0 for k in top_k_list},
        'mrr':       {k: 0.0 for k in top_k_list},
        'type_p':    {k: 0.0 for k in top_k_list},
        'vibe_p':    {k: 0.0 for k in top_k_list},
        'tt_align':  {k: 0.0 for k in top_k_list},
        'slot_cov':  {k: 0.0 for k in top_k_list},
        'n_users':   0,
    }


def _topk_desc(scores, k):
    if len(scores) <= k:
        idx = np.arange(len(scores))
    else:
        idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(scores[idx])[::-1]]


def _ranking_metrics(topk_ids, gt_ids, k):
    hits = 0
    ap = 0.0
    dcg = 0.0
    rr = 0.0
    for rank, pid in enumerate(topk_ids[:k], start=1):
        if pid in gt_ids:
            hits += 1
            ap += hits / rank
            dcg += 1.0 / np.log2(rank + 1)
            if rr == 0.0:
                rr = 1.0 / rank
    denom = min(len(gt_ids), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, denom + 1))
    return {
        'hits': hits,
        'ap': ap / denom if denom else 0.0,
        'ndcg': dcg / idcg if idcg > 0 else 0.0,
        'mrr': rr,
    }


def _update_metric_acc(acc, top_k_list, ret_ids, ret_tt, ret_cats, gt_ids, gt_types, gt_vibes, uintent, biz_meta):
    acc['n_users'] += 1
    for k in top_k_list:
        topk_ids  = ret_ids[:k]
        topk_tt   = ret_tt[:k]
        topk_cats = ret_cats[:k]

        rank_m = _ranking_metrics(topk_ids, gt_ids, k)
        hits = rank_m['hits']
        acc['hit_rate'][k]  += float(hits > 0)
        acc['precision'][k] += hits / k
        acc['recall'][k]    += hits / len(gt_ids)
        acc['map_k'][k]     += rank_m['ap']
        acc['ndcg'][k]      += rank_m['ndcg']
        acc['mrr'][k]       += rank_m['mrr']

        tm = sum(1 for bid in topk_ids if gt_types & biz_meta.get(bid, {}).get('types', set()))
        vm = sum(1 for bid in topk_ids if gt_vibes & biz_meta.get(bid, {}).get('vibes', set()))
        acc['type_p'][k] += tm / k
        acc['vibe_p'][k] += vm / k

        if uintent:
            acc['tt_align'][k] += float(np.sum(topk_tt == uintent)) / k

        slot_types_found = {category_to_slot(cat) for cat in topk_cats} - {''}
        acc['slot_cov'][k] += len(REQUIRED_SLOTS & slot_types_found) / len(REQUIRED_SLOTS)


def _acc_to_rows(acc, top_k_list, retrieval_scope, user_segment):
    rows = []
    d = max(acc['n_users'], 1)
    for k in top_k_list:
        rows.append({
            'RetrievalScope'       : retrieval_scope,
            'UserSegment'          : user_segment,
            'K'                    : k,
            'Users'                : acc['n_users'],
            'Precision@K'          : acc['precision'][k] / d,
            'Recall@K'             : acc['recall'][k]    / d,
            'MAP@K'                : acc['map_k'][k]     / d,
            'NDCG@K'               : acc['ndcg'][k]      / d,
            'MRR@K'                : acc['mrr'][k]       / d,
            'HitRate@K'            : acc['hit_rate'][k]  / d,
            'Type_Precision@K'     : acc['type_p'][k]    / d,
            'Vibe_Precision@K'     : acc['vibe_p'][k]    / d,
            'TravelType_Align@K'   : acc['tt_align'][k]  / d,
            'SlotCoverage@K'       : acc['slot_cov'][k]  / d,
        })
    return rows


def compute_user_level_metrics(df_test, query_tower, candidate_tower, top_k_list, df_candidates, vocab):
    id_col  = 'id' if 'id' in df_candidates.columns else 'business_id'
    emb_col = 'bge_embedding' if 'bge_embedding' in df_candidates.columns else 'embedding'

    print("1. Extracting candidate embeddings ...")
    cand_dict = {
        'business_id':  tf.constant(df_candidates[id_col].astype(str).values, dtype=tf.string),
        'city':         tf.constant(df_candidates['city'].astype(str).values, dtype=tf.string),
        'category':     tf.constant(df_candidates['category'].astype(str).values, dtype=tf.string),
        'travel_type':  tf.constant(df_candidates['travel_type'].fillna('').astype(str).values, dtype=tf.string),
        'types':        tf.constant([pad_sequence(lst, Config.MAX_HISTORY_LEN) for lst in df_candidates['types']], dtype=tf.string),
        'vibes':        tf.constant([pad_sequence(lst, Config.MAX_HISTORY_LEN) for lst in df_candidates['vibes']], dtype=tf.string),
        'stars_biz':    tf.constant(df_candidates['stars_biz'].fillna(0).astype(np.float32).values, dtype=tf.float32),
        'review_count': tf.constant(df_candidates['review_count'].fillna(0).astype(np.float32).values, dtype=tf.float32),
        'semantic_emb': tf.constant(np.vstack(df_candidates[emb_col].values), dtype=tf.float32),
    }
    all_cand_embs   = candidate_tower(cand_dict, training=False).numpy()
    all_cand_ids    = df_candidates[id_col].astype(str).values
    all_cand_cities = df_candidates['city'].astype(str).values
    all_cand_tt     = df_candidates['travel_type'].fillna('').astype(str).values
    all_cand_cats   = df_candidates['category'].fillna('').astype(str).values

    print("2. Building metadata lookup ...")
    biz_meta = {}
    for _, row in df_candidates.iterrows():
        bid = str(row.get('id', row.get('business_id', '')))
        biz_meta[bid] = {
            'types': set(row['types']) if isinstance(row['types'], (list, np.ndarray)) else set(),
            'vibes': set(row['vibes']) if isinstance(row['vibes'], (list, np.ndarray)) else set(),
            'travel_type': str(row.get('travel_type', '') or ''),
            'category':    str(row.get('category', '')),
        }

    print("3. Preparing user-level queries ...")
    id_col_test = 'id' if 'id' in df_test.columns else 'business_id'
    gt_dict = df_test.groupby('user_id')[id_col_test].apply(lambda s: set(s.astype(str))).to_dict()
    df_q    = df_test.groupby('user_id').first().reset_index()

    test_q_dict = {
        'user_id':             tf.constant(df_q['user_id'].astype(str).values, dtype=tf.string),
        'current_city':        tf.constant(df_q['current_city'].astype(str).values, dtype=tf.string),
        'trip_intent':         tf.constant(df_q['trip_intent'].fillna('').astype(str).values, dtype=tf.string),
        'intent_vibe':         tf.constant(df_q['intent_vibe'].fillna('').astype(str).values, dtype=tf.string),
        'history_types':       tf.constant([pad_sequence(lst, Config.MAX_HISTORY_LEN) for lst in df_q['history_types']], dtype=tf.string),
        'history_vibes':       tf.constant([pad_sequence(lst, Config.MAX_HISTORY_LEN) for lst in df_q['history_vibes']], dtype=tf.string),
        'history_business_id': tf.constant([pad_sequence(lst, Config.MAX_HISTORY_LEN) for lst in df_q['history_business_id']], dtype=tf.string),
    }
    ds_test_q = tf.data.Dataset.from_tensor_slices(test_q_dict).batch(Config.BATCH_SIZE)

    max_k       = max(top_k_list)
    vocab_users = set(vocab['user_id'])
    scopes      = ['city_aware', 'global']
    segments    = ['all', 'loyal', 'near_cold']
    accs = {(scope, segment): _empty_metric_acc(top_k_list) for scope in scopes for segment in segments}

    print("4. Evaluating users (city-aware and global retrieval) ...")
    for batch in ds_test_q:
        q_embs       = query_tower(batch, training=False).numpy()
        user_ids     = batch['user_id'].numpy()
        user_cities  = batch['current_city'].numpy()
        user_intents = batch['trip_intent'].numpy()

        for i in range(len(user_ids)):
            uid      = user_ids[i].decode('utf-8')
            ucity    = user_cities[i].decode('utf-8')
            uintent  = user_intents[i].decode('utf-8')
            u_emb    = q_embs[i]
            segment  = 'loyal' if uid in vocab_users else 'near_cold'

            gt_ids = gt_dict.get(uid, set())
            if not gt_ids:
                continue

            gt_types = set(); gt_vibes = set()
            for gid in gt_ids:
                gt_types.update(biz_meta.get(gid, {}).get('types', set()))
                gt_vibes.update(biz_meta.get(gid, {}).get('vibes', set()))

            for scope in scopes:
                if scope == 'city_aware':
                    cand_idx = np.where(all_cand_cities == ucity)[0]
                    if len(cand_idx) == 0:
                        continue
                else:
                    cand_idx = np.arange(len(all_cand_ids))

                scores  = np.dot(all_cand_embs[cand_idx], u_emb)
                top_idx = _topk_desc(scores, max_k)
                chosen  = cand_idx[top_idx]

                ret_ids  = all_cand_ids[chosen]
                ret_tt   = all_cand_tt[chosen]
                ret_cats = all_cand_cats[chosen]

                _update_metric_acc(accs[(scope, 'all')], top_k_list, ret_ids, ret_tt, ret_cats,
                                   gt_ids, gt_types, gt_vibes, uintent, biz_meta)
                _update_metric_acc(accs[(scope, segment)], top_k_list, ret_ids, ret_tt, ret_cats,
                                   gt_ids, gt_types, gt_vibes, uintent, biz_meta)

    rows = []
    for scope in scopes:
        for segment in segments:
            rows.extend(_acc_to_rows(accs[(scope, segment)], top_k_list, scope, segment))

    print('\n--- Users evaluated by scope/segment ---')
    for scope in scopes:
        for segment in segments:
            print(f'  {scope:10s} | {segment:9s}: {accs[(scope, segment)]["n_users"]:,}')

    return pd.DataFrame(rows), accs[('city_aware', 'all')]['n_users']


print('Computing user-level evaluation metrics ...')
t0 = time.time()
df_metrics, n_eval = compute_user_level_metrics(
    df_test, query_tower, candidate_tower, Config.TOP_K_LIST, df_candidates, vocab
)
print(f'Evaluated {n_eval:,} city-aware users in {time.time()-t0:.1f}s')
print('\nEvaluation Results:')
print(df_metrics.to_string(index=False, float_format='{:.4f}'.format))

csv_path = str(MODEL_SAVE_DIR / 'eval_results_user_level_v15.csv')
df_metrics.to_csv(csv_path, index=False)
print(f'Saved -> {csv_path}')


Computing user-level evaluation metrics ...
1. Extracting candidate embeddings ...
2. Building metadata lookup ...
3. Preparing user-level queries ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


4. Evaluating users (city-aware and global retrieval) ...

--- Users evaluated by scope/segment ---
  city_aware | all      : 76,881
  city_aware | loyal    : 45,221
  city_aware | near_cold: 31,660
  global     | all      : 76,881
  global     | loyal    : 45,221
  global     | near_cold: 31,660
Evaluated 76,881 city-aware users in 2135.6s

Evaluation Results:
RetrievalScope UserSegment   K  Users  Precision@K  Recall@K  MAP@K  NDCG@K  MRR@K  HitRate@K  Type_Precision@K  Vibe_Precision@K  TravelType_Align@K  SlotCoverage@K
    city_aware         all  10  76881       0.0259    0.2345 0.1014  0.1351 0.1106     0.2566            0.2949            0.9640              0.6405          0.7461
    city_aware         all  50  76881       0.0106    0.4680 0.1124  0.1878 0.1223     0.5110            0.2953            0.9306              0.6771          0.8882
    city_aware         all 100  76881       0.0068    0.5926 0.1143  0.2088 0.1241     0.6431            0.2778            0.8837         

In [ ]:
# Cell 5d -- Visualization (city-aware/all primary metrics)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Plot primary production-like scope. Full table still contains global + user segments.
df_plot = df_metrics[
    (df_metrics['RetrievalScope'] == 'city_aware') &
    (df_metrics['UserSegment'] == 'all')
].copy()

k_values = df_plot['K'].tolist()
x        = np.arange(len(k_values))
width    = 0.18

group1 = ['Precision@K', 'Recall@K', 'HitRate@K']
group2 = ['MAP@K', 'NDCG@K', 'MRR@K']
group3 = ['Type_Precision@K', 'Vibe_Precision@K']
group4 = ['TravelType_Align@K', 'SlotCoverage@K']

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.suptitle(f'Two-Tower v15 -- City-aware User-level Evaluation\n'
             f'(n_users={n_eval:,}, temporal 80/10/10 split)',
             fontsize=15, fontweight='bold')


def plot_group(ax, metrics, title, ylim=None):
    colors = plt.cm.Set2(np.linspace(0, 1, len(metrics)))
    for i, (metric, color) in enumerate(zip(metrics, colors)):
        if metric not in df_plot.columns:
            continue
        vals = df_plot[metric].tolist()
        bars = ax.bar(x + i * width, vals, width, label=metric, color=color)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(x + width * (len(metrics) - 1) / 2)
    ax.set_xticklabels([f'K={k}' for k in k_values], fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    if ylim:
        ax.set_ylim(0, ylim)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.legend(fontsize=9, loc='upper left')


plot_group(axes[0, 0], group1, 'Retrieval Coverage',      ylim=0.75)
plot_group(axes[0, 1], group2, 'Ranking Quality',         ylim=0.5)
plot_group(axes[1, 0], group3, 'Semantic Match',          ylim=1.15)
plot_group(axes[1, 1], group4, 'Travel Intent & Slots',   ylim=1.15)

plt.tight_layout()
eval_plot_path = str(MODEL_SAVE_DIR / 'eval_metrics_v15.png')
plt.savefig(eval_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Evaluation chart saved -> {eval_plot_path}')


Evaluation chart saved -> /content/drive/MyDrive/train/output_train_15/dim256_hist30_attn_temp00500/eval_metrics_v15.png


### Test

In [ ]:
# Cell 6 -- Inference: Slot-quota diverse retrieval (v2)
# Cai thien: MMR, sub-quota attraction, entertainment theo travel_type, Pub/Bar
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle

# ── Hang so (local override) ──────────────────────────────────────────────────
# Tong: 10 diem/ngay; entertainment 1->2, cafe 2->1
_DAILY_QUOTA = {
    "attraction":    4,
    "restaurant":    3,
    "cafe":          1,
    "entertainment": 2,
    "accommodation": 1,   # 1 lua chon/ngay (2 ngay -> 2 options) -- cap o max 3
}

TONG_HOP_INTENT = 'Khám phá tổng hợp'
KNOWN_TRAVEL_TYPES_FOR_DIV = [
    'Ẩm thực & Bản địa',
    'Đô thị & Vui chơi',
    'Khám phá & Sinh thái',
    'Nghỉ dưỡng & Biển',
    'Văn hóa & Lịch sử',
]

# 0.0=chi diversity, 1.0=chi relevance; 0.7 can bang tot cho travel rec
MMR_LAMBDA = 0.7

# Entertainment pool theo travel_type -- filter bang types list (bat Pub/Bar, Spa)
# primary: thu truoc; secondary: fallback khi primary khong du candidates
ENTERTAINMENT_BY_TRAVEL_TYPE = {
    # primary   : phù hợp nhất với chủ đề hành trình (chỉ dùng Giải trí & Thư giãn & Thể thao)
    # secondary : giải trí phổ thông — fallback khi primary không đủ
    # last_resort: pub/bar/karaoke/billiards — chỉ khi primary+secondary < 2x quota
    # NOTE: Không đưa attraction types vào đây (đã xử lý ở slot attraction)
    'Văn hóa & Lịch sử': {
        'primary':    ['Nhà hát/Sân khấu', 'Bảo tàng nghệ thuật/3D', 'Bảo tàng & Không gian trưng bày'],
        'secondary':  ['Rạp phim', 'Thể thao ngoài trời', 'Thể thao trong nhà'],
        'last_resort':['Karaoke', 'Pub/Bar', 'Billiards'],
    },
    'Khám phá & Sinh thái': {
        'primary':    ['Thể thao ngoài trời', 'Công viên giải trí', 'Thể thao trong nhà'],
        'secondary':  ['Rạp phim', 'Bảo tàng & Không gian trưng bày'],
        'last_resort':['Karaoke', 'Pub/Bar', 'Billiards'],
    },
    'Nghỉ dưỡng & Biển': {
        'primary':    ['Spa & Thư giãn', 'Thể thao ngoài trời', 'Thể thao trong nhà'],
        'secondary':  ['Rạp phim', 'Nhà hát/Sân khấu'],
        'last_resort':['Pub/Bar', 'Karaoke', 'Billiards'],
    },
    'Đô thị & Vui chơi': {
        'primary':    ['Rạp phim', 'Công viên giải trí', 'Nhà hát/Sân khấu', 'Bảo tàng nghệ thuật/3D'],
        'secondary':  ['Thể thao ngoài trời', 'Thể thao trong nhà', 'Spa & Thư giãn'],
        'last_resort':['Karaoke', 'Billiards', 'Pub/Bar'],
    },
    'Ẩm thực & Bản địa': {
        'primary':    ['Rạp phim', 'Nhà hát/Sân khấu'],
        'secondary':  ['Bảo tàng nghệ thuật/3D', 'Thể thao ngoài trời', 'Spa & Thư giãn'],
        'last_resort':['Pub/Bar', 'Karaoke', 'Billiards'],
    },
    'Khám phá tổng hợp': {
        'primary':    ['Rạp phim', 'Công viên giải trí', 'Nhà hát/Sân khấu', 'Spa & Thư giãn'],
        'secondary':  ['Thể thao ngoài trời', 'Bảo tàng nghệ thuật/3D',
                       'Bảo tàng & Không gian trưng bày', 'Thể thao trong nhà'],
        'last_resort':['Karaoke', 'Pub/Bar', 'Billiards'],
    },
}

# ── Lưu trú pool ─────────────────────────────────────────────────────────────
_ACCOMMODATION_TYPES = {'Khách sạn & Resort', 'Homestay & Villa', 'Nhà nghỉ'}

def _get_accommodation_pool(df_biz, city, used_ids, id_col):
    city_mask = df_biz['city'] == city
    used_mask = ~df_biz[id_col].isin(used_ids)
    pool = df_biz[
        city_mask & used_mask &
        df_biz['types'].apply(lambda t: _has_any_type(t, _ACCOMMODATION_TYPES))
    ].copy()
    # Uu tien chat luong: stars cao truoc, cung stars thi nhieu review hon
    if len(pool) > 0:
        pool = pool.sort_values(['stars_biz', 'review_count'],
                                ascending=[False, False]).reset_index(drop=True)
    return pool


# Sub-quota attraction: tranh qua nhieu diem cung loai (vd 4 chua/den)
# [(type_name, max_per_day), ...] -- fill theo thu tu uu tien, dung khi du limit
# Phase 2 tu dong fill slot con lai bang MMR tren phan chua duoc chon
ATTRACTION_TYPE_QUOTA = {
    'Văn hóa & Lịch sử': [
        ('Công trình tôn giáo',             2),  # chua/den/nha tho: toi da 2
        ('Di tích',                          1),
        ('Bảo tàng & Không gian trưng bày',  1),
        ('Làng nghề',                        1),
    ],
    'Khám phá & Sinh thái': [
        ('Thiên nhiên',         2),  # rung/nui/thac: toi da 2
        ('Nông trại',           1),
        ('Bãi biển/Vịnh',       1),
        ('Tour có hướng dẫn',   1),
    ],
    'Nghỉ dưỡng & Biển': [
        ('Bãi biển/Vịnh',               3),  # beach trip: nhieu bai bien hop ly
        ('Đài quan sát & Khu chụp ảnh', 1),  # sunset viewpoint
    ],
    'Đô thị & Vui chơi': [
        ('Đài quan sát & Khu chụp ảnh', 1),
        ('Công viên/Quảng trường',       1),
        ('Công viên giải trí',           1),  # theme park
        ('Trung tâm thương mại',         1),  # cap: 1 mall du roi
        # Karaoke/Rap phim: khong co trong quota -> chi xuat hien o Phase 2 neu con slot
    ],
    'Ẩm thực & Bản địa': [
        ('Chợ truyền thống',                2),  # cho dia phuong: 2/chuyen
        ('Cửa hàng đặc sản/Quà lưu niệm',  1),
        ('Phố đi bộ',                       1),
    ],
}

# Loai tru khoi attraction pool (khong phai dia diem tham quan thuc su)
_EXCLUDE_FROM_ATTRACTION = {
    'Homestay & Villa', 'Khách sạn & Resort', 'Nhà nghỉ',
    'Dịch vụ du lịch', 'Cửa hàng tiện lợi',
}


# ── Tien ich ─────────────────────────────────────────────────────────────────

def pad_sequence(lst, max_len, pad_value=''):
    if not isinstance(lst, (list, np.ndarray)): lst = []
    lst = list(lst)[-max_len:]
    if len(lst) < max_len: lst = [pad_value] * (max_len - len(lst)) + lst
    return lst


def _has_any_type(types_val, type_set):
    if isinstance(types_val, (list, np.ndarray)):
        return any(str(x) in type_set for x in types_val)
    return str(types_val) in type_set


def _encode_pool(pool_df):
    if len(pool_df) == 0:
        return np.zeros((0, Config.OUTPUT_DIM), np.float32), np.array([])
    id_col  = 'id' if 'id' in pool_df.columns else 'business_id'
    emb_col = 'bge_embedding' if 'bge_embedding' in pool_df.columns else 'embedding'
    pool_df = pool_df.reset_index(drop=True)
    embs = []
    bs = Config.BATCH_SIZE
    for start in range(0, len(pool_df), bs):
        chunk = pool_df.iloc[start:start + bs]
        cand = {
            'business_id':  tf.constant(chunk[id_col].astype(str).values,                            dtype=tf.string),
            'city':         tf.constant(chunk['city'].astype(str).values,                             dtype=tf.string),
            'category':     tf.constant(chunk['category'].astype(str).values,                         dtype=tf.string),
            'travel_type':  tf.constant(chunk['travel_type'].fillna('').astype(str).values,           dtype=tf.string),
            'types':        tf.constant([pad_sequence(lst, Config.MAX_HISTORY_LEN) for lst in chunk['types']], dtype=tf.string),
            'vibes':        tf.constant([pad_sequence(lst, Config.MAX_HISTORY_LEN) for lst in chunk['vibes']], dtype=tf.string),
            'stars_biz':    tf.constant(chunk['stars_biz'].fillna(0).astype(np.float32).values,       dtype=tf.float32),
            'review_count': tf.constant(chunk['review_count'].fillna(0).astype(np.float32).values,    dtype=tf.float32),
            'semantic_emb': tf.constant(np.vstack(chunk[emb_col].values),                             dtype=tf.float32),
        }
        embs.append(candidate_tower(cand, training=False).numpy())
    return np.concatenate(embs, axis=0), pool_df[id_col].values


def mmr_rerank(cand_embs, pool, query_emb, limit, lambda_=MMR_LAMBDA):
    # Maximum Marginal Relevance: greedy chon item co cao diem MMR nhat
    # MMR(i) = lambda * relevance(i) - (1-lambda) * max_sim(i, selected)
    if len(pool) == 0 or len(cand_embs) == 0:
        return pool.iloc[0:0].copy()
    scores   = np.dot(cand_embs, query_emb)
    norms    = np.linalg.norm(cand_embs, axis=1, keepdims=True)
    norms    = np.where(norms < 1e-8, 1e-8, norms)
    embs_n   = cand_embs / norms
    selected  = []
    remaining = list(range(len(pool)))
    for _ in range(min(limit, len(pool))):
        if not selected:
            best = remaining[int(np.argmax([scores[i] for i in remaining]))]
        else:
            sel_embs        = embs_n[selected]
            best, best_mmr  = remaining[0], -np.inf
            for idx in remaining:
                rel     = scores[idx]
                max_sim = float(np.max(embs_n[idx] @ sel_embs.T))
                score   = lambda_ * rel - (1 - lambda_) * max_sim
                if score > best_mmr:
                    best_mmr, best = score, idx
        selected.append(best)
        remaining.remove(best)
    pool             = pool.reset_index(drop=True)
    result           = pool.iloc[selected].copy()
    result['score']  = np.array(scores)[selected]
    return result


# ── Pool helpers ──────────────────────────────────────────────────────────────

def _get_base_pool(df_biz, city, slot_type, travel_type, id_col):
    city_mask = df_biz['city'] == city
    if slot_type == 'attraction':
        if travel_type == TONG_HOP_INTENT:
            mask = city_mask
        else:
            mask = city_mask & (df_biz['travel_type'].fillna('') == travel_type)
        # Loai tru: khach san, dich vu du lich, tien loi -- khong phai diem tham quan
        mask = mask & ~df_biz['types'].apply(
            lambda t: _has_any_type(t, _EXCLUDE_FROM_ATTRACTION))
    elif slot_type == 'restaurant':
        mask = city_mask & df_biz['category'].str.contains(
            'thực|Thực|thuc|Am thuc', na=False, regex=True)
    elif slot_type == 'cafe':
        def _is_cafe(t):
            if isinstance(t, (list, np.ndarray)):
                return any(str(x) in {'Cafe & Đồ uống', 'Tiệm bánh & Tráng miệng'} or
                           'cafe' in str(x).lower() or 'đồ uống' in str(x).lower()
                           for x in t)
            return False
        mask = city_mask & df_biz['types'].apply(_is_cafe)
    else:
        return pd.DataFrame()
    return df_biz[mask].copy()


def _get_entertainment_pool(df_biz, city, travel_type, used_ids, id_col):
    config      = ENTERTAINMENT_BY_TRAVEL_TYPE.get(
        travel_type, ENTERTAINMENT_BY_TRAVEL_TYPE['Khám phá tổng hợp'])
    primary     = set(config['primary'])
    secondary   = set(config.get('secondary', []))
    last_resort = set(config.get('last_resort', []))
    city_mask   = df_biz['city'] == city
    used_mask   = ~df_biz[id_col].isin(used_ids)

    pool_pri = df_biz[
        city_mask & used_mask &
        df_biz['types'].apply(lambda t: _has_any_type(t, primary))
    ].copy()

    already  = set(pool_pri[id_col].tolist()) if len(pool_pri) else set()
    pool_sec = df_biz[
        city_mask & used_mask &
        df_biz['types'].apply(lambda t: _has_any_type(t, secondary)) &
        ~df_biz[id_col].isin(already)
    ].copy()

    pool = pd.concat([pool_pri, pool_sec], ignore_index=True)

    # last_resort (pub/bar, karaoke, billiards): chi them khi primary+secondary
    # chua du candidates de MMR chon (thuong xay ra o thanh pho nho)
    _min_pool = _DAILY_QUOTA.get('entertainment', 2) * 2
    if len(pool) < _min_pool and last_resort:
        already2 = set(pool[id_col].tolist()) if len(pool) else set()
        pool_lr  = df_biz[
            city_mask & used_mask &
            df_biz['types'].apply(lambda t: _has_any_type(t, last_resort)) &
            ~df_biz[id_col].isin(already2)
        ].copy()
        if len(pool_lr) > 0:
            pool = pd.concat([pool, pool_lr], ignore_index=True)
            print(f'    [ent] Pool nho ({len(pool)-len(pool_lr)}), fallback +{len(pool_lr)} last_resort')

    # Cap toi da 3 dia diem moi type de tranh mot type chiem qua nhieu
    if len(pool) > 0:
        def _first_type(t):
            return str(t[0]) if isinstance(t, list) and len(t) > 0 else ''
        pool = (pool.groupby(pool['types'].apply(_first_type), group_keys=False)
                    .apply(lambda g: g.head(3))
                    .reset_index(drop=True))
    return pool


def _get_attraction_with_subquota(pool, query_emb, limit, num_days, travel_type, id_col):
    type_quota = ATTRACTION_TYPE_QUOTA.get(travel_type, [])
    if not type_quota:
        embs, _ = _encode_pool(pool)
        return mmr_rerank(embs, pool, query_emb, limit)

    collected  = []
    used_local = set()
    remaining  = limit

    # Phase 1: fill tung type theo quota uu tien
    for type_name, per_day_max in type_quota:
        if remaining <= 0:
            break
        type_limit = min(per_day_max * num_days, remaining)
        type_pool  = pool[
            pool['types'].apply(lambda t: _has_any_type(t, {type_name})) &
            ~pool[id_col].isin(used_local)
        ].copy()
        if len(type_pool) == 0:
            continue
        embs, _ = _encode_pool(type_pool)
        rows     = mmr_rerank(embs, type_pool, query_emb, type_limit)
        collected.append(rows)
        used_local.update(rows[id_col].tolist())
        remaining -= len(rows)

    # Phase 2: dien slot con lai tu pool chua duoc chon (MMR tren tat ca)
    if remaining > 0:
        leftover = pool[~pool[id_col].isin(used_local)].copy()
        if len(leftover) > 0:
            embs, _ = _encode_pool(leftover)
            collected.append(mmr_rerank(embs, leftover, query_emb, remaining))

    if not collected:
        return pd.DataFrame()
    return pd.concat(collected, ignore_index=True)


def _diverse_attraction_topk(pool, query_emb, limit, id_col):
    # Khi TONG_HOP: chia quota deu cho 5 travel_type, dung MMR trong moi loai
    n_types = len(KNOWN_TRAVEL_TYPES_FOR_DIV)
    quotas  = {tt: limit // n_types for tt in KNOWN_TRAVEL_TYPES_FOR_DIV}
    extra   = limit % n_types

    type_best = {}
    for tt in KNOWN_TRAVEL_TYPES_FOR_DIV:
        sub = pool[pool['travel_type'].fillna('') == tt].copy()
        if len(sub) == 0:
            type_best[tt] = -np.inf
            continue
        embs, _ = _encode_pool(sub)
        type_best[tt] = float(np.dot(embs, query_emb).max()) if len(embs) else -np.inf

    for tt in sorted(KNOWN_TRAVEL_TYPES_FOR_DIV, key=lambda x: type_best[x], reverse=True)[:extra]:
        quotas[tt] += 1

    selected = []
    for tt in KNOWN_TRAVEL_TYPES_FOR_DIV:
        q = quotas[tt]
        if q == 0:
            continue
        sub = pool[pool['travel_type'].fillna('') == tt].copy()
        if len(sub) == 0:
            continue
        embs, _ = _encode_pool(sub)
        if len(embs) == 0:
            continue
        sub  = sub.reset_index(drop=True)
        rows = mmr_rerank(embs, sub, query_emb, q)
        selected.append(rows)

    if not selected:
        return pd.DataFrame()
    return pd.concat(selected, ignore_index=True)


# Guard: Bootstrap cell phai duoc chay truoc cell nay.
# Flow: Cell 03 -> 04 -> 05 -> 22 -> Bootstrap -> [cell nay]
if "candidate_tower" not in globals() or "model" not in globals():
    raise RuntimeError(
        "Thieu bien. Hay chay Bootstrap cell truoc cell nay.")

# Reload best weights
model.load_weights(str(MODEL_SAVE_DIR / "best_model.weights.h5"))
print("Best weights loaded.")



# ── Main inference ─────────────────────────────────────────────────────────────

def retrieve_diverse_topk(city, travel_type, num_days=2,
                           user_id='test_user',
                           history_types=[], history_vibes=[], history_biz=[],
                           intent_vibe=''):
    print(f'\n{"="*60}')
    print(f'  City: {city}  |  Travel type: {travel_type}  |  {num_days} ngay')
    print(f'  User: {user_id}  |  Vibe: {intent_vibe or "(default)"}')
    if travel_type == TONG_HOP_INTENT:
        print(f'  [Kham pha tong hop] Attraction: lay deu tu 5 loai hanh trinh')
    print(f'{"="*60}')

    id_col   = 'id' if 'id' in df_candidates.columns else 'business_id'
    results  = []
    used_ids = set()

    is_new_user = user_id not in set(vocab['user_id'])
    print(f'  {"New user (OOV)" if is_new_user else "Returning user"} -- '
          f'{"city + travel_type" if is_new_user else "full history vector"} lam query.')

    q_dict = {
        'user_id':             tf.constant([user_id],          dtype=tf.string),
        'current_city':        tf.constant([city],             dtype=tf.string),
        'trip_intent':         tf.constant([travel_type],      dtype=tf.string),
        'intent_vibe':         tf.constant([intent_vibe],      dtype=tf.string),
        'history_types':       tf.constant([pad_sequence(history_types, Config.MAX_HISTORY_LEN)], dtype=tf.string),
        'history_vibes':       tf.constant([pad_sequence(history_vibes, Config.MAX_HISTORY_LEN)], dtype=tf.string),
        'history_business_id': tf.constant([pad_sequence(history_biz,   Config.MAX_HISTORY_LEN)], dtype=tf.string),
    }
    query_emb = query_tower(q_dict, training=False).numpy()[0]

    quota = {slot: cnt * num_days for slot, cnt in _DAILY_QUOTA.items()}

    for slot_type, limit in quota.items():
        # Lay pool candidates cho slot nay
        if slot_type == 'entertainment':
            pool = _get_entertainment_pool(df_candidates, city, travel_type, used_ids, id_col)
        elif slot_type == 'accommodation':
            pool = _get_accommodation_pool(df_candidates, city, used_ids, id_col)
        else:
            pool = _get_base_pool(df_candidates, city, slot_type, travel_type, id_col)
            pool = pool[~pool[id_col].isin(used_ids)].copy()

        if len(pool) == 0:
            print(f'  [{slot_type:14s}] Pool rong -- bo qua.')
            continue

        # Score & rank
        if slot_type == 'attraction' and travel_type == TONG_HOP_INTENT:
            top_rows = _diverse_attraction_topk(pool, query_emb, limit, id_col)
        elif slot_type == 'attraction':
            top_rows = _get_attraction_with_subquota(pool, query_emb, limit, num_days, travel_type, id_col)
        else:
            cand_embs, _ = _encode_pool(pool)
            # Entertainment uu tien diversity hon (lambda thap hon)
            lam      = max(0.5, MMR_LAMBDA - 0.2) if slot_type == 'entertainment' else MMR_LAMBDA
            top_rows = mmr_rerank(cand_embs, pool, query_emb, limit, lambda_=lam)

        if len(top_rows) == 0:
            continue

        print(f'\n  [{slot_type.upper():14s}] Top-{limit}:')
        for rank, (_, biz) in enumerate(top_rows.iterrows(), 1):
            score_str  = f'{biz.get("score", 0):.3f}'
            tt_str     = f' | tt={biz.get("travel_type", "?")}' if travel_type == TONG_HOP_INTENT else ''
            types_val  = biz.get('types', [])
            first_type = types_val[0] if isinstance(types_val, list) and len(types_val) > 0 else str(types_val)
            print(f'    {rank}. [{score_str}] {biz.get("name", "?")} '
                  f'| Stars:{biz.get("stars_biz", 0):.1f} '
                  f'({int(biz.get("review_count", 0))} reviews)'
                  f' | {first_type}{tt_str}')
            used_ids.add(biz[id_col])
            results.append(biz.to_dict())

    print(f'\n  Tong dia diem goi y: {len(results)} (cho {num_days} ngay)')
    print(f'  -> Ready for itinerary planner module.')
    return results


# ── Test ─────────────────────────────────────────────────────────────────────

retrieve_diverse_topk(
    city        = 'Vũng Tàu',
    travel_type = 'Khám phá & Sinh thái',
    num_days    = 2,
    user_id     = '45',
    intent_vibe = '',
)


Best weights loaded.

  City: Vũng Tàu  |  Travel type: Khám phá & Sinh thái  |  2 ngay
  User: 45  |  Vibe: (default)
  Returning user -- full history vector lam query.

  [ATTRACTION    ] Top-8:
    1. [0.813] Hồ Mây Park - Vũng Tàu | Stars:3.6 (75 reviews) | Thiên nhiên
    2. [0.638] Đồi Con Heo | Stars:3.8 (22 reviews) | Thiên nhiên
    3. [0.500] Khu Du Lịch Suối Đá | Stars:3.5 (11 reviews) | Thiên nhiên
    4. [0.490] Khu Du Lịch Bình Châu - Hồ Cốc | Stars:3.8 (8 reviews) | Thiên nhiên
    5. [0.638] Nông Trại Cừu - Đồng Cừu Châu Pha | Stars:3.9 (18 reviews) | Nông trại
    6. [0.731] Vũng Tàu Marina - Bến Du Thuyền | Stars:3.6 (43 reviews) | Tour có hướng dẫn
    7. [0.476] Thiền Tôn Phật Quang - Núi Dinh | Stars:4.3 (6 reviews) | Thiên nhiên
    8. [0.475] Đèo Nước Ngọt | Stars:3.5 (8 reviews) | Thiên nhiên

  [RESTAURANT    ] Top-6:
    1. [0.849] Ốc Tự Nhiên - Trần Phú | Stars:3.7 (293 reviews) | Quán ăn
    2. [0.788] Hải Sản Gành Hào - Trần Phú | Stars:3.7 (474 reviews) | 

[{'name': 'Hồ Mây Park - Vũng Tàu',
  'address': 'Hồ Mây, Phường Vũng Tàu, Thành phố Hồ Chí Minh',
  'city': 'Vũng Tàu',
  'latitude': 10.35935,
  'longitude': 107.068285,
  'stars_biz': 3.6,
  'review_count': 75,
  'category': 'Tham quan & Khám phá',
  'types': ['Thiên nhiên'],
  'vibes': ['Ngoài trời',
   'Yên tĩnh',
   'View đẹp',
   'Khách solo',
   'Gia đình & Trẻ em'],
  'travel_type': 'Khám phá & Sinh thái',
  'combined_text': 'Hồ Mây Park - Vũng Tàu. Thiên nhiên tại Vũng Tàu.',
  'unified_id': 'ad94d992-913b-5714-841f-63ce3baed3b9',
  'bge_embedding': [0.0232893024,
   0.0376564078,
   -0.0810107514,
   -0.0121583622,
   -0.0385133997,
   -0.0171674602,
   -0.001431366,
   0.034302365,
   0.0279337931,
   0.0014192066,
   0.0038311828,
   -0.0395028964,
   0.0005812398,
   -0.0040335846,
   0.0003586018,
   -0.0237019844,
   0.0400489196,
   0.0344251394,
   0.008167145800000001,
   0.0080516562,
   -0.0026704604,
   0.0013767029000000001,
   0.0362259001,
   -0.0197176337,
   

### Tự động ngắt kết nối Colab (Tiết kiệm Compute Units)

In [ ]:
# import time
# from google.colab import runtime

# print("Tất cả các tác vụ đã hoàn thành. Đang chuẩn bị ngắt kết nối runtime để tiết kiệm Colab Pro...")
# # Đợi 5 giây để các log cuối cùng kịp ghi lại
# time.sleep(20)

# # Lệnh ngắt kết nối và giải phóng tài nguyên
# runtime.unassign()